<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/TADP_Scalability_Experiment_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [ ]:
#!/usr/bin/env python3
# =============================================================================
# TADP EXPERIMENT C v3.5 FAST + MEMORY-SAFE — ALL-IN-ONE / ONE-CELL / THREE-SEED VERSION
# =============================================================================
#
# Design:
#   K = [10, 20, 30, 50]
#   seeds = [42, 142, 242]
#   7 scenarios
#   4 FL rounds
#   1 local epoch
#
# Internally:
#   - each K×seed block runs in a fresh Python process;
#   - evidence profiles S1-S10 are sampled uniformly WITH replacement;
#   - evidence assignment is seeded, reproducible and nested across K;
#   - no target admission percentage is imposed;
#   - Random-K matches same-run TADP-VR Federated client count;
#   - Random-K matches exact same-round optimizer-step budget;
#   - centralized↔federated optimizer-step parity is enforced;
#   - final validation and manuscript summaries run automatically.
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path
import numpy as np

MASTER_KS = [10, 20, 30, 50]
MASTER_SEEDS = [42, 142, 242]

CORE_SOURCE = '#!/usr/bin/env python3\n# =============================================================================\n# TADP EXPERIMENT C — END-TO-END CLIENT-SCALE SCALABILITY\n# Corrected publication-safe version\n# =============================================================================\n#\n# Purpose\n# -------\n# Evaluate end-to-end scalability as the number of distributed contributors\n# increases while the total dataset and learning task remain fixed.\n#\n# K = {10, 20, 30, 50}\n# Seeds = {42, 142, 242, 342, 442}\n# FL rounds = 4\n# Local epochs = 1\n# Batch size = 64\n#\n# Seven scenarios:\n#   Centralized\n#     1. Naïve Centralized\n#     2. TADP-AA Centralized\n#     3. TADP-VR Centralized\n#\n#   Federated\n#     4. Vanilla FedAvg\n#     5. TADP-AA Federated\n#     6. TADP-VR Federated\n#     7. Random-K\n#\n# Key controls\n# ------------\n# 1. Fixed held-out test set across every K and seed.\n# 2. Three-class target: NO=0, <30=1, >30=2.\n# 3. Client partition occurs only inside the fixed global TRAIN set.\n# 4. Shared scaler is obtained from aggregated TRAIN-only sufficient statistics.\n# 5. No K-dependent numeric drift is injected.\n# 6. TADP-VR is based on HPS + WAC + zero-score safeguard.\n# 7. WAC minimum is fixed at 0.70.\n# 8. S1-S10 documentary evidence profiles are randomly sampled with replacement.\n# 9. Random profile assignment is seeded, reproducible, and nested across K.\n# 10. DQ is measured from each actual client TRAIN partition.\n# 11. TADP-VR centralized and federated use the same same-seed frozen cohort.\n# 12. Random-K matches same-run TADP-VR Federated:\n#       - exact client count in every round;\n#       - exact total optimizer-step budget in every round.\n# 13. Centralized/federated pairs use exact total optimizer-step parity.\n# 14. Full-participation baselines really use all K clients.\n# 15. Centralized communication is N/A; no arbitrary 15 MB is invented.\n# 16. FL communication uses actual FP32 model size × participants × 2 × 1.12.\n# 17. No artificial RAM floors and no artificial storage floors.\n#\n# This is a client-scale/federation-size scalability experiment, not a\n# big-data scalability experiment, because the total dataset is held fixed.\n# =============================================================================\n\nimport os\nimport gc\nimport json\nimport math\nimport random\nimport time\nimport threading\nimport warnings\nimport sys\nimport ctypes\nimport faulthandler\nimport traceback\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nwarnings.filterwarnings("ignore")\nos.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"\nos.environ["CUDA_VISIBLE_DEVICES"] = "-1"\n\nimport numpy as np\nimport pandas as pd\nimport psutil\nimport matplotlib.pyplot as plt\n\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import (\n    accuracy_score,\n    precision_score,\n    recall_score,\n    f1_score,\n    roc_auc_score,\n)\n\nimport tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import layers\n\n\n# =============================================================================\n# 1. CONFIGURATION\n# =============================================================================\n\nEXPERIMENT_VERSION = "TADP-EXPC-END2END-v3.5-FAST-MEMSAFE-RANDOM-EVIDENCE-3SEEDS"\n\nDATA_PATH = Path("./data_cache/diabetes_130US.csv")\nOUTDIR = Path("./experiment_C_end_to_end_v3_5_fast_memsafe_random_evidence_3seeds")\nTABLES_DIR = OUTDIR / "tables"\nFIG_DIR = TABLES_DIR / "figs"\nLEDGER_DIR = OUTDIR / "ledgers"\n\nKS = [10, 20, 30, 50]\nRUN_SEEDS = [42, 142, 242]\n\nNUM_ROUNDS_FL = 4\nLOCAL_EPOCHS = 1\nBATCH_SIZE = 64\nLEARNING_RATE = 1e-3\nDIRICHLET_ALPHA = 1.0\nPROTOCOL_OVERHEAD = 0.12\n\n# Estimated energy only; not hardware-metered.\nPOWER_W = 12.0\nCOST_USD_PER_KWH = 0.20\nCARBON_KG_PER_KWH = 0.475\n\n# TADP policy\nGOOD_CUT = 3.0\nHIGH_CUT = 3.5\nWAC_MIN = 0.70\nADEQUATE_FACTOR_SCORE = 3.0\nZERO_SCORE_SAFEGUARD = True\n\nWEIGHTS = {\n    "dim1": 0.25,  # Source Reliability\n    "dim2": 0.15,  # Data Quality\n    "dim3": 0.10,  # Documentation\n    "dim4": 0.10,  # Timeliness\n    "dim5": 0.30,  # Regulatory Compliance\n    "dim6": 0.10,  # Context / Usage\n}\n\nSCENARIOS = [\n    "Naïve Centralized",\n    "TADP-AA Centralized",\n    "TADP-VR Centralized",\n    "Vanilla FedAvg",\n    "TADP-AA Federated",\n    "TADP-VR Federated",\n    "Random-K",\n]\n\nFED_SCENARIOS = [\n    "Vanilla FedAvg",\n    "TADP-AA Federated",\n    "TADP-VR Federated",\n    "Random-K",\n]\n\nPAIR_MAP = {\n    "Naïve Centralized": "Vanilla FedAvg",\n    "TADP-AA Centralized": "TADP-AA Federated",\n    "TADP-VR Centralized": "TADP-VR Federated",\n}\n\n# Re-run only blocks that are incomplete or from another experiment version.\nRESUME = True\n\n\n# -------------------------------------------------------------------------\n# Fresh-process block execution controls.\n#\n# The RunAll launcher sets TADP_ONLY_K and TADP_ONLY_SEED so each K×seed block\n# executes in a fresh Python/TensorFlow process. TADP_POST_ONLY=1 skips training\n# and performs only the final global validation / summaries.\n# -------------------------------------------------------------------------\n_ONLY_K_ENV = os.getenv("TADP_ONLY_K")\n_ONLY_SEED_ENV = os.getenv("TADP_ONLY_SEED")\nWORKER_MODE = (_ONLY_K_ENV is not None) or (_ONLY_SEED_ENV is not None)\nPOST_ONLY = os.getenv("TADP_POST_ONLY", "0") == "1"\n\nif (_ONLY_K_ENV is None) ^ (_ONLY_SEED_ENV is None):\n    raise RuntimeError(\n        "Worker mode requires both TADP_ONLY_K and TADP_ONLY_SEED."\n    )\n\nACTIVE_KS = [int(_ONLY_K_ENV)] if WORKER_MODE else list(KS)\nACTIVE_SEEDS = [int(_ONLY_SEED_ENV)] if WORKER_MODE else list(RUN_SEEDS)\n\nif WORKER_MODE:\n    if ACTIVE_KS[0] not in KS:\n        raise RuntimeError(f"Unsupported worker K={ACTIVE_KS[0]}; expected one of {KS}.")\n    if ACTIVE_SEEDS[0] not in RUN_SEEDS:\n        raise RuntimeError(\n            f"Unsupported worker seed={ACTIVE_SEEDS[0]}; expected one of {RUN_SEEDS}."\n        )\n\n# Random documentary-evidence assignment.\n# A master profile sequence is generated once per seed for C001..C050.\n# K=10/20/30/50 use prefixes of the same seed-specific sequence.\nPROFILE_ASSIGNMENT_SEED_BASE = 731_951\nPROFILE_ASSIGNMENT_MODE = "seeded_uniform_random_with_replacement_nested_by_seed"\n\nfor p in [OUTDIR, TABLES_DIR, FIG_DIR, LEDGER_DIR]:\n    p.mkdir(parents=True, exist_ok=True)\n\nWORKER_ERROR_LOG = OUTDIR / "worker_error.log"\ntry:\n    faulthandler.enable()\nexcept Exception:\n    pass\n\n\n# =============================================================================\n# 2. REPRODUCIBILITY\n# =============================================================================\n\ndef set_seed(seed: int):\n    seed = int(seed)\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        tf.keras.utils.set_random_seed(seed)\n    except Exception:\n        tf.random.set_seed(seed)\n    try:\n        tf.config.experimental.enable_op_determinism()\n    except Exception:\n        pass\n\n\n# =============================================================================\n# 3. PROCESS RSS MONITOR\n# =============================================================================\n\ndef release_keras_memory():\n    """\n    Release TensorFlow/Keras graph state and Python heap objects between\n    short-lived local client models. This changes memory handling only.\n    """\n    try:\n        tf.keras.backend.clear_session()\n    except Exception:\n        pass\n\n    gc.collect()\n\n    try:\n        ctypes.CDLL("libc.so.6").malloc_trim(0)\n    except Exception:\n        pass\n\n\n\nclass PeakRSS:\n    def __init__(self, interval_s=0.05):\n        self.interval_s = float(interval_s)\n        self.proc = psutil.Process(os.getpid())\n        self.start_rss = 0\n        self.peak_rss = 0\n        self.stop_event = threading.Event()\n        self.thread = None\n\n    def start(self):\n        self.start_rss = int(self.proc.memory_info().rss)\n        self.peak_rss = self.start_rss\n        self.stop_event.clear()\n\n        def _loop():\n            while not self.stop_event.is_set():\n                try:\n                    rss = int(self.proc.memory_info().rss)\n                    self.peak_rss = max(self.peak_rss, rss)\n                except Exception:\n                    pass\n                self.stop_event.wait(self.interval_s)\n\n        self.thread = threading.Thread(target=_loop, daemon=True)\n        self.thread.start()\n        return self\n\n    def stop(self):\n        self.stop_event.set()\n        if self.thread is not None:\n            self.thread.join(timeout=1.0)\n        try:\n            self.peak_rss = max(self.peak_rss, int(self.proc.memory_info().rss))\n        except Exception:\n            pass\n        return {\n            "rss_start_mb": self.start_rss / (1024 ** 2),\n            "rss_peak_mb": self.peak_rss / (1024 ** 2),\n            "rss_peak_delta_mb": max(0.0, (self.peak_rss - self.start_rss) / (1024 ** 2)),\n        }\n\n\n# =============================================================================\n# 4. DATA PREPARATION\n# =============================================================================\n\ndef preprocess_diabetes(df: pd.DataFrame) -> pd.DataFrame:\n    """\n    Three-class task:\n      NO   -> 0\n      <30  -> 1\n      >30  -> 2\n    """\n    df = df.copy()\n    df.columns = (\n        df.columns\n        .astype(str)\n        .str.lower()\n        .str.replace(r"[^a-z0-9]+", "_", regex=True)\n    )\n    df.replace("?", np.nan, inplace=True)\n\n    if "age" in df.columns and df["age"].dtype == object:\n        df["age"] = pd.to_numeric(\n            df["age"].astype(str).str.extract(r"(\\d+)", expand=False),\n            errors="coerce",\n        )\n\n    if "readmitted" not in df.columns:\n        raise RuntimeError("Dataset is missing required target column \'readmitted\'.")\n\n    mapping = {"NO": 0, "NONE": 0, "<30": 1, ">30": 2}\n    target_text = df["readmitted"].astype(str).str.strip().str.upper()\n    mapped = target_text.map(mapping)\n\n    if mapped.isna().any():\n        unknown = sorted(target_text[mapped.isna()].unique().tolist())[:20]\n        raise RuntimeError(f"Unrecognized readmitted labels: {unknown}")\n\n    df["readmitted"] = mapped.astype(np.int32)\n\n    # Patient identifiers must not become model features.\n    df.drop(columns=["encounter_id", "patient_nbr"], errors="ignore", inplace=True)\n\n    return df\n\n\nif not DATA_PATH.exists():\n    raise FileNotFoundError(\n        f"Dataset not found at {DATA_PATH}. "\n        "Upload/copy diabetes_130US.csv into ./data_cache/ first."\n    )\n\nRAW = pd.read_csv(DATA_PATH)\nDATA = preprocess_diabetes(RAW)\nDATASET_ROWS = int(len(DATA))\ndel RAW\ngc.collect()\n\n# Fixed global split for the entire Experiment C.\nGLOBAL_TRAIN_DF, GLOBAL_TEST_DF = train_test_split(\n    DATA,\n    test_size=0.20,\n    random_state=42,\n    stratify=DATA["readmitted"],\n)\nGLOBAL_TRAIN_DF = GLOBAL_TRAIN_DF.reset_index(drop=True)\nGLOBAL_TEST_DF = GLOBAL_TEST_DF.reset_index(drop=True)\n\ndel DATA\ngc.collect()\n\nNUMERIC_COLS = (\n    GLOBAL_TRAIN_DF\n    .drop(columns=["readmitted"])\n    .select_dtypes(include=[np.number])\n    .columns\n    .tolist()\n)\n\nif not NUMERIC_COLS:\n    raise RuntimeError("No numeric model features are available.")\n\nprint("=" * 96)\nprint("TADP EXPERIMENT C — END-TO-END CLIENT-SCALE SCALABILITY")\nprint("=" * 96)\nprint(f"Version: {EXPERIMENT_VERSION}")\nprint(f"Dataset rows: {DATASET_ROWS:,}")\nprint(f"Fixed global TRAIN: {len(GLOBAL_TRAIN_DF):,}")\nprint(f"Fixed global TEST:  {len(GLOBAL_TEST_DF):,}")\nprint(f"Numeric model features: {len(NUMERIC_COLS)}")\nprint(f"K values: {KS}")\nprint(f"Seeds: {RUN_SEEDS}")\nprint(f"Scenarios: {SCENARIOS}")\nprint(\n    "Global test classes:",\n    GLOBAL_TEST_DF["readmitted"].value_counts().sort_index().to_dict()\n)\nprint("=" * 96)\n\n\n# =============================================================================\n# 5. CLIENT PARTITION\n# =============================================================================\n\ndef make_client_ids(k: int):\n    return [f"C{i:03d}" for i in range(1, int(k) + 1)]\n\n\ndef dirichlet_partition(frame: pd.DataFrame, k: int, seed: int, alpha: float):\n    """\n    Partition only GLOBAL TRAIN by class using a fixed Dirichlet alpha.\n    No K-dependent numeric shift is added.\n    """\n    ids = make_client_ids(k)\n    labels = frame["readmitted"].to_numpy()\n    rng = np.random.default_rng(int(seed) + 77001 + 31 * int(k))\n    buckets = {cid: [] for cid in ids}\n\n    for cls in sorted(np.unique(labels)):\n        idx = np.where(labels == cls)[0]\n        rng.shuffle(idx)\n\n        proportions = rng.dirichlet(np.repeat(float(alpha), int(k)))\n        cuts = (np.cumsum(proportions) * len(idx)).astype(int)\n        parts = np.split(idx, cuts[:-1])\n\n        for cid, part in zip(ids, parts):\n            buckets[cid].extend(part.tolist())\n\n    # Repair any empty logical client deterministically.\n    empty = [cid for cid in ids if len(buckets[cid]) == 0]\n    for cid in empty:\n        donor = max(ids, key=lambda c: len(buckets[c]))\n        if len(buckets[donor]) <= 1:\n            raise RuntimeError("Unable to repair empty client partition.")\n        buckets[cid].append(buckets[donor].pop())\n\n    flattened = [r for cid in ids for r in buckets[cid]]\n    if len(flattened) != len(frame) or len(set(flattened)) != len(frame):\n        raise RuntimeError(f"K={k}: partition is not complete/disjoint.")\n\n    raw_clients = {\n        cid: frame.iloc[sorted(buckets[cid])].copy().reset_index(drop=True)\n        for cid in ids\n    }\n    return ids, raw_clients\n\n\n# =============================================================================\n# 6. TRAIN-ONLY FEDERATED SCALER\n# =============================================================================\n\ndef aggregate_train_scaler(raw_clients):\n    """\n    Aggregate local count, sum and sum-of-squares over TRAIN only.\n    No raw client frames are concatenated for scaler fitting.\n    """\n    d = len(NUMERIC_COLS)\n    total_count = np.zeros(d, dtype=np.float64)\n    total_sum = np.zeros(d, dtype=np.float64)\n    total_sumsq = np.zeros(d, dtype=np.float64)\n\n    for frame in raw_clients.values():\n        arr = frame[NUMERIC_COLS].to_numpy(dtype=np.float64)\n        valid = np.isfinite(arr)\n        safe = np.where(valid, arr, 0.0)\n\n        total_count += valid.sum(axis=0)\n        total_sum += safe.sum(axis=0)\n        total_sumsq += np.square(safe).sum(axis=0)\n\n    denom = np.maximum(total_count, 1.0)\n    mean = total_sum / denom\n    var = np.maximum(total_sumsq / denom - np.square(mean), 0.0)\n    scale = np.sqrt(var)\n    scale[~np.isfinite(scale) | (scale < 1e-12)] = 1.0\n    return mean, scale\n\n\ndef transform_frame(frame, mean, scale):\n    arr = frame[NUMERIC_COLS].to_numpy(dtype=np.float64)\n    z = (arr - mean) / scale\n    return np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)\n\n\ndef prepare_k_seed(k: int, seed: int):\n    ids, raw_clients = dirichlet_partition(\n        GLOBAL_TRAIN_DF, k=k, seed=seed, alpha=DIRICHLET_ALPHA\n    )\n    mean, scale = aggregate_train_scaler(raw_clients)\n\n    client_arrays = {}\n    for cid in ids:\n        X = transform_frame(raw_clients[cid], mean, scale)\n        y = raw_clients[cid]["readmitted"].to_numpy(dtype=np.int32)\n        client_arrays[cid] = (X, y)\n\n    Xtest = transform_frame(GLOBAL_TEST_DF, mean, scale)\n    ytest = GLOBAL_TEST_DF["readmitted"].to_numpy(dtype=np.int32)\n\n    if set(np.unique(ytest).tolist()) != {0, 1, 2}:\n        raise RuntimeError("Fixed global test set is not a valid three-class test set.")\n\n    return ids, raw_clients, client_arrays, Xtest, ytest\n\n\n# =============================================================================\n# 7. TRUST FACTORS\n# =============================================================================\n\nFACTOR_NAMES = {\n    "dim1": ["source_reputation", "data_controller", "data_objective"],\n    "dim2": ["completeness", "duplication_rate", "error_rate", "type_consistency"],\n    "dim3": ["data_dictionary", "version_logs", "collection_protocol", "definition_updates"],\n    "dim4": ["data_freshness", "scheduled_refresh", "retention_clarity"],\n    "dim5": [\n        "regulation_coverage", "consent_ethics", "geo_restrictions",\n        "sensitivity_classification", "audits"\n    ],\n    "dim6": ["license_terms", "ethical_reviews", "redistribution", "user_agreements"],\n}\n\nCONTROLLED_EVIDENCE_PROFILES = {\n    "S1": {\n        "description": "Strong, consistently supported documentary evidence",\n        "factors": {\n            "dim1": [5, 5, 4],\n            "dim3": [5, 4, 5, 4],\n            "dim4": [4, 5, 4],\n            "dim5": [5, 5, 4, 5, 5],\n            "dim6": [5, 4, 5, 4],\n        },\n    },\n    "S2": {\n        "description": "Strong evidence with limited version-history support",\n        "factors": {\n            "dim1": [4, 4, 5],\n            "dim3": [4, 2, 3, 3],\n            "dim4": [4, 4, 4],\n            "dim5": [4, 4, 4, 5, 4],\n            "dim6": [4, 4, 4, 4],\n        },\n    },\n    "S3": {\n        "description": "Strong governance evidence with weaker timeliness support",\n        "factors": {\n            "dim1": [4, 4, 4],\n            "dim3": [4, 4, 4, 3],\n            "dim4": [3, 2, 2],\n            "dim5": [4, 4, 4, 4, 4],\n            "dim6": [4, 4, 4, 3],\n        },\n    },\n    "S4": {\n        "description": "No validated accountable-controller evidence",\n        "factors": {\n            "dim1": [3, 0, 3],\n            "dim3": [4, 4, 4, 4],\n            "dim4": [4, 4, 4],\n            "dim5": [4, 4, 4, 4, 4],\n            "dim6": [4, 4, 4, 4],\n        },\n    },\n    "S5": {\n        "description": "Required consent/ethics evidence is absent",\n        "factors": {\n            "dim1": [4, 4, 4],\n            "dim3": [4, 4, 4, 4],\n            "dim4": [4, 4, 4],\n            "dim5": [4, 0, 4, 4, 4],\n            "dim6": [4, 4, 4, 4],\n        },\n    },\n    "S6": {\n        "description": "Explicit user-agreement non-compliance",\n        "factors": {\n            "dim1": [4, 4, 4],\n            "dim3": [4, 4, 4, 4],\n            "dim4": [4, 4, 4],\n            "dim5": [4, 4, 4, 4, 4],\n            "dim6": [4, 4, 4, 0],\n        },\n    },\n    "S7": {\n        "description": "Compensation-heavy profile with broad low adequacy",\n        "factors": {\n            "dim1": [5, 2, 2],\n            "dim3": [5, 5, 2, 2],\n            "dim4": [5, 2, 2],\n            "dim5": [5, 5, 2, 2, 2],\n            "dim6": [5, 5, 2, 2],\n        },\n    },\n    "S8": {\n        "description": "Broadly adequate evidence with several limited factors",\n        "factors": {\n            "dim1": [3, 3, 4],\n            "dim3": [3, 2, 3, 3],\n            "dim4": [3, 3, 2],\n            "dim5": [3, 3, 3, 2, 3],\n            "dim6": [3, 3, 2, 3],\n        },\n    },\n    "S9": {\n        "description": "Consistently adequate documentary evidence",\n        "factors": {\n            "dim1": [3, 3, 3],\n            "dim3": [3, 3, 3, 3],\n            "dim4": [3, 3, 3],\n            "dim5": [3, 3, 3, 3, 3],\n            "dim6": [3, 3, 3, 3],\n        },\n    },\n    "S10": {\n        "description": "Multiple weak documentary factors without zero",\n        "factors": {\n            "dim1": [2, 2, 3],\n            "dim3": [2, 2, 3, 2],\n            "dim4": [2, 3, 2],\n            "dim5": [2, 2, 3, 2, 2],\n            "dim6": [2, 2, 3, 2],\n        },\n    },\n}\n\n\n@lru_cache(maxsize=None)\ndef master_evidence_profile_assignment(seed: int):\n    """\n    Reproducible random documentary-evidence assignment for one experimental seed.\n\n    Design:\n      - profiles S1..S10 are sampled uniformly WITH replacement;\n      - no admission ratio is targeted;\n      - no S1..S10 cyclic repetition is used;\n      - one master sequence is generated for C001..C050;\n      - K=10,20,30,50 use prefixes of that same seed-specific sequence.\n\n    Therefore profile composition varies naturally, while comparisons across K\n    within a seed remain nested rather than being regenerated independently.\n    """\n    seed = int(seed)\n    profile_ids = sorted(\n        CONTROLLED_EVIDENCE_PROFILES.keys(),\n        key=lambda s: int(s[1:])\n    )\n    rng = np.random.default_rng(PROFILE_ASSIGNMENT_SEED_BASE + seed)\n    draws = rng.choice(\n        profile_ids,\n        size=max(KS),\n        replace=True\n    ).tolist()\n\n    return {\n        f"C{i:03d}": str(draws[i - 1])\n        for i in range(1, max(KS) + 1)\n    }\n\n\ndef evidence_profile_for_client(cid: str, seed: int):\n    mapping = master_evidence_profile_assignment(int(seed))\n    if cid not in mapping:\n        raise KeyError(f"No randomized evidence profile assignment for client {cid}.")\n    sid = mapping[cid]\n    return sid, CONTROLLED_EVIDENCE_PROFILES[sid]\n\n\ndef evidence_assignment_table(k: int, seed: int):\n    mapping = master_evidence_profile_assignment(int(seed))\n    ids = make_client_ids(int(k))\n    rows = []\n    for cid in ids:\n        sid = mapping[cid]\n        rows.append({\n            "experiment_version": EXPERIMENT_VERSION,\n            "K": int(k),\n            "seed": int(seed),\n            "client": cid,\n            "evidence_profile": sid,\n            "profile_description": CONTROLLED_EVIDENCE_PROFILES[sid]["description"],\n            "assignment_mode": PROFILE_ASSIGNMENT_MODE,\n            "assignment_seed": int(PROFILE_ASSIGNMENT_SEED_BASE + int(seed)),\n        })\n    return pd.DataFrame(rows)\n\n\ndef score_missing(x):\n    x = max(0.0, float(x))\n    if x > 50: return 0.0\n    if x >= 20: return 1.0\n    if x >= 10: return 2.0\n    if x >= 5: return 3.0\n    if x >= 1: return 4.0\n    return 5.0\n\n\ndef score_duplication(x):\n    x = max(0.0, float(x))\n    if x < 2: return 5.0\n    if x <= 5: return 3.0\n    if x <= 10: return 2.0\n    if x <= 20: return 1.0\n    return 0.0\n\n\ndef score_error(x):\n    x = max(0.0, float(x))\n    if x > 15: return 0.0\n    if x >= 10: return 1.0\n    if x >= 5: return 2.0\n    if x >= 2: return 3.0\n    if x >= 1: return 4.0\n    return 5.0\n\n\ndef score_type_consistency(x):\n    x = float(np.clip(x, 0.0, 100.0))\n    if x == 0: return 5.0\n    if x <= 5: return 4.0\n    if x <= 15: return 3.0\n    if x <= 30: return 2.0\n    if x <= 50: return 1.0\n    return 0.0\n\n\ndef dq_factor_scores(client_df: pd.DataFrame):\n    feat = client_df.drop(columns=["readmitted"], errors="ignore").copy()\n\n    total_cells = max(1, feat.shape[0] * max(1, feat.shape[1]))\n    missing_pct = 100.0 * feat.isna().sum().sum() / total_cells\n    dup_pct = 100.0 * feat.duplicated().mean() if len(feat) else 100.0\n\n    numeric = feat.select_dtypes(include=[np.number])\n    if numeric.size:\n        arr = numeric.to_numpy(dtype=np.float64)\n        error_pct = 100.0 * np.isinf(arr).sum() / max(1, arr.size)\n    else:\n        error_pct = 0.0\n\n    inconsistent = 0\n    for c in feat.columns:\n        vals = feat[c].dropna()\n        if len(vals) == 0:\n            continue\n        observed = {type(v).__name__ for v in vals.iloc[: min(2000, len(vals))]}\n        if len(observed) > 1:\n            inconsistent += 1\n\n    inconsistent_pct = 100.0 * inconsistent / max(1, feat.shape[1])\n\n    return {\n        "completeness": score_missing(missing_pct),\n        "duplication_rate": score_duplication(dup_pct),\n        "error_rate": score_error(error_pct),\n        "type_consistency": score_type_consistency(inconsistent_pct),\n    }\n\n\ndef build_factor_scores(cid: str, client_df: pd.DataFrame, seed: int):\n    sid, profile = evidence_profile_for_client(cid, seed)\n    factors = {}\n\n    for dim in ["dim1", "dim3", "dim4", "dim5", "dim6"]:\n        values = profile["factors"][dim]\n        names = FACTOR_NAMES[dim]\n        factors[dim] = {\n            name: float(value)\n            for name, value in zip(names, values)\n        }\n\n    factors["dim2"] = dq_factor_scores(client_df)\n    return sid, profile["description"], factors\n\n\ndef dimension_scores(factors):\n    return {\n        dim: float(np.mean(list(vals.values())))\n        for dim, vals in factors.items()\n    }\n\n\ndef calculate_hps(dims):\n    return float(sum(WEIGHTS[d] * dims[d] for d in WEIGHTS))\n\n\ndef calculate_wac(factors):\n    coverage = 0.0\n    for dim, weight in WEIGHTS.items():\n        vals = [float(v) for v in factors[dim].values()]\n        adequate = sum(v >= ADEQUATE_FACTOR_SCORE for v in vals) / len(vals)\n        coverage += weight * adequate\n    return float(coverage)\n\n\ndef zero_score_factors(factors):\n    return [\n        f"{dim}.{name}"\n        for dim, vals in factors.items()\n        for name, value in vals.items()\n        if np.isfinite(float(value)) and float(value) <= 0.0\n    ]\n\n\ndef policy_decision(hps: float, wac: float, zeros):\n    if ZERO_SCORE_SAFEGUARD and zeros:\n        return "ZERO_SCORE_SAFEGUARD", "QUARANTINE", "factor score 0"\n\n    if hps < GOOD_CUT:\n        return "QUARANTINE", "QUARANTINE", "HPS below review threshold"\n\n    if hps < HIGH_CUT:\n        passed = wac >= WAC_MIN\n        return (\n            "REVIEW",\n            "ACCEPT" if passed else "REJECT",\n            "Review WAC pass" if passed else "Review WAC fail",\n        )\n\n    passed = wac >= WAC_MIN\n    return (\n        "AUTO_ACCEPT" if passed else "ADEQUACY_CHECK",\n        "ACCEPT" if passed else "REJECT",\n        "Direct HPS+WAC pass" if passed else "High-HPS WAC fail",\n    )\n\n\ndef run_governance(raw_clients, k, seed, scenario_name):\n    """\n    Evaluate the same evidence and same policy for every governed scenario.\n    TADP-AA later keeps all clients, but the policy output is still retained\n    for auditability.\n    """\n    t0 = time.perf_counter()\n    rows = []\n\n    scenario_ledger_dir = LEDGER_DIR / f"K{k}" / f"seed_{seed}"\n    scenario_ledger_dir.mkdir(parents=True, exist_ok=True)\n\n    safe_name = (\n        scenario_name\n        .replace(" ", "_")\n        .replace("/", "_")\n        .replace("ï", "i")\n    )\n    ledger_path = scenario_ledger_dir / f"{safe_name}.jsonl"\n\n    # Each invocation has its own actual ledger file.\n    if ledger_path.exists():\n        ledger_path.unlink()\n\n    bytes_written = 0\n\n    with open(ledger_path, "ab") as f:\n        for cid, client_df in raw_clients.items():\n            sid, desc, factors = build_factor_scores(cid, client_df, seed)\n            dims = dimension_scores(factors)\n            hps = calculate_hps(dims)\n            wac = calculate_wac(factors)\n            zeros = zero_score_factors(factors)\n            initial, final, reason = policy_decision(hps, wac, zeros)\n\n            row = {\n                "experiment_version": EXPERIMENT_VERSION,\n                "K": int(k),\n                "seed": int(seed),\n                "scenario": scenario_name,\n                "client": cid,\n                "evidence_profile": sid,\n                "evidence_description": desc,\n                "evidence_assignment_mode": PROFILE_ASSIGNMENT_MODE,\n                "evidence_assignment_seed": int(PROFILE_ASSIGNMENT_SEED_BASE + int(seed)),\n                "hps": hps,\n                "wac": wac,\n                "zero_factors": ",".join(zeros),\n                "initial_action": initial,\n                "policy_final_action": final,\n                "reason": reason,\n                **{f"{d}_score": dims[d] for d in sorted(dims)},\n            }\n            rows.append(row)\n\n            payload = (json.dumps(row, sort_keys=True, separators=(",", ":")) + "\\n").encode("utf-8")\n            f.write(payload)\n            bytes_written += len(payload)\n\n    elapsed = time.perf_counter() - t0\n    gov = pd.DataFrame(rows)\n    accepted = gov.loc[gov["policy_final_action"].eq("ACCEPT"), "client"].tolist()\n\n    return gov, accepted, elapsed, int(bytes_written), str(ledger_path)\n\n\n# =============================================================================\n# 8. MODEL\n# =============================================================================\n\ndef build_model(input_dim: int):\n    inp = keras.Input(shape=(input_dim,), dtype=tf.float32)\n    x = layers.Dense(128, activation="relu")(inp)\n    x = layers.Dropout(0.30)(x)\n    x = layers.Dense(64, activation="relu")(x)\n    x = layers.Dropout(0.30)(x)\n    x = layers.Dense(64, activation="relu")(x)\n    x = layers.Dropout(0.20)(x)\n    x = layers.Dense(64, activation="relu")(x)\n    x = layers.Dropout(0.20)(x)\n    x = layers.Dense(64, activation="relu")(x)\n    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)\n\n    model = keras.Model(inp, out)\n    model.compile(\n        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),\n        loss="sparse_categorical_crossentropy",\n        metrics=["accuracy"],\n    )\n    return model\n\n\ndef prepare_reusable_local_model(input_dim: int):\n    """\n    Create one compiled local model for a federated scenario.\n\n    The model object is reused across clients to avoid repeated TensorFlow graph\n    creation. Before every client, both model weights and Adam optimizer state\n    are reset, which is equivalent to starting that client with a fresh local\n    optimizer while keeping memory bounded.\n    """\n    model = build_model(input_dim)\n\n    # Materialize Adam slot variables once so their initial state can be restored.\n    try:\n        model.optimizer.build(model.trainable_variables)\n    except Exception:\n        # A harmless zero-size setup is avoided; the first train call would\n        # otherwise create the slots. Current Keras versions support build().\n        pass\n\n    opt_initial_state = []\n    try:\n        opt_initial_state = [v.numpy().copy() for v in model.optimizer.variables]\n    except Exception:\n        opt_initial_state = []\n\n    return model, opt_initial_state\n\n\ndef reset_optimizer_state(model, initial_state):\n    """\n    Restore the optimizer to its initial state before a new client trains.\n    This resets Adam\'s iteration counter and moment estimates.\n    """\n    variables = list(model.optimizer.variables)\n\n    if initial_state and len(variables) == len(initial_state):\n        for var, value in zip(variables, initial_state):\n            var.assign(value)\n        return\n\n    # Fallback for an optimizer whose slots were created lazily after first use.\n    # Iteration/moment variables are reset while the learning-rate variable is\n    # preserved.\n    for var in variables:\n        name = str(getattr(var, "name", "")).lower()\n        if "learning_rate" in name or "learning-rate" in name:\n            continue\n        try:\n            var.assign(tf.zeros_like(var))\n        except Exception:\n            pass\n\n\n\ndef evaluate_model(model, Xtest, ytest):\n    probs = model.predict(Xtest, verbose=0)\n    if probs.ndim != 2 or probs.shape[1] != 3:\n        raise RuntimeError(f"Expected three output probabilities; got {probs.shape}")\n\n    yhat = np.argmax(probs, axis=1)\n\n    return {\n        "accuracy": float(accuracy_score(ytest, yhat)),\n        "precision": float(precision_score(ytest, yhat, average="macro", zero_division=0)),\n        "recall": float(recall_score(ytest, yhat, average="macro", zero_division=0)),\n        "f1": float(f1_score(ytest, yhat, average="macro", zero_division=0)),\n        "roc_auc": float(\n            roc_auc_score(ytest, probs, multi_class="ovr", average="macro")\n        ),\n    }\n\n\n# =============================================================================\n# 9. OPTIMIZER-STEP BUDGET\n# =============================================================================\n\ndef natural_client_steps(client_arrays, cid):\n    n = len(client_arrays[cid][0])\n    return max(1, math.ceil(n / BATCH_SIZE) * LOCAL_EPOCHS)\n\n\ndef train_exact_steps(model, X, y, steps, seed):\n    """\n    Train for the exact optimizer-step budget with bounded input-pipeline memory.\n    """\n    steps = int(steps)\n    if steps <= 0:\n        return 0.0\n\n    ds = tf.data.Dataset.from_tensor_slices((X, y))\n    ds = ds.shuffle(\n        buffer_size=min(len(X), 10000),\n        seed=int(seed),\n        reshuffle_each_iteration=True,\n    )\n    ds = ds.repeat().batch(BATCH_SIZE, drop_remainder=False).prefetch(1)\n\n    options = tf.data.Options()\n    try:\n        options.autotune.enabled = False\n    except Exception:\n        pass\n    ds = ds.with_options(options)\n\n    t0 = time.perf_counter()\n    model.fit(ds, epochs=1, steps_per_epoch=steps, verbose=0)\n    elapsed = float(time.perf_counter() - t0)\n\n    del ds\n    gc.collect()\n    return elapsed\n\n\ndef allocate_exact_randomk_steps(natural_step_map, target_total):\n    """\n    Exact Random-K round budget:\n      sum allocated steps == same-round TADP-VR optimizer steps.\n    """\n    ids = list(natural_step_map)\n    if not ids:\n        raise RuntimeError("Random-K received an empty client set.")\n\n    target_total = int(target_total)\n    if target_total < len(ids):\n        raise RuntimeError(\n            f"Target {target_total} steps is smaller than Random-K cohort size {len(ids)}."\n        )\n\n    natural = np.asarray(\n        [max(1, int(natural_step_map[c])) for c in ids],\n        dtype=float,\n    )\n\n    raw = target_total * natural / natural.sum()\n    alloc = np.floor(raw).astype(int)\n    alloc = np.maximum(alloc, 1)\n\n    while int(alloc.sum()) < target_total:\n        remainder = raw - alloc\n        idx = int(np.argmax(remainder))\n        alloc[idx] += 1\n\n    while int(alloc.sum()) > target_total:\n        candidates = np.where(alloc > 1)[0]\n        if len(candidates) == 0:\n            raise RuntimeError("Cannot reduce Random-K allocation to exact target.")\n        idx = int(candidates[np.argmax(alloc[candidates] - raw[candidates])])\n        alloc[idx] -= 1\n\n    result = {cid: int(v) for cid, v in zip(ids, alloc)}\n\n    if sum(result.values()) != target_total:\n        raise RuntimeError(\n            f"Random-K exact-step allocation failed: "\n            f"{sum(result.values())} != {target_total}"\n        )\n\n    return result\n\n\n# =============================================================================\n# 10. FEDAVG AGGREGATION\n# =============================================================================\n\ndef aggregate_weights(local_weights, sample_counts):\n    if not local_weights:\n        raise RuntimeError("No local weights to aggregate.")\n\n    total = float(sum(sample_counts))\n    if total <= 0:\n        raise RuntimeError("Invalid FedAvg sample count.")\n\n    out = []\n    for layer_idx in range(len(local_weights[0])):\n        acc = np.zeros_like(local_weights[0][layer_idx], dtype=np.float64)\n\n        for weights, n in zip(local_weights, sample_counts):\n            acc += weights[layer_idx].astype(np.float64) * (float(n) / total)\n\n        out.append(acc.astype(local_weights[0][layer_idx].dtype))\n\n    return out\n\n\n# =============================================================================\n# 11. SCENARIO SEEDS\n# =============================================================================\n\ndef pair_initialization_seed(seed, scenario):\n    """\n    Common randomization within the comparison families.\n\n    Full-data family:\n      Naïve Centralized / Vanilla FedAvg / TADP-AA Centralized /\n      TADP-AA Federated share the same initialization seed. This makes the\n      baseline-vs-AA difference primarily a governance-overhead comparison.\n\n    Selected-client family:\n      TADP-VR Centralized / TADP-VR Federated / Random-K share the same\n      initialization seed, so identity selection rather than initialization\n      is the main varying factor.\n    """\n    if scenario in [\n        "Naïve Centralized",\n        "Vanilla FedAvg",\n        "TADP-AA Centralized",\n        "TADP-AA Federated",\n    ]:\n        return int(seed) + 1000\n\n    if scenario in [\n        "TADP-VR Centralized",\n        "TADP-VR Federated",\n        "Random-K",\n    ]:\n        return int(seed) + 3000\n\n    return int(seed) + 9000\n\n\n# =============================================================================\n# 12. CENTRALIZED SCENARIO\n# =============================================================================\n\ndef run_centralized(\n    scenario,\n    k,\n    seed,\n    client_ids,\n    raw_clients,\n    client_arrays,\n    Xtest,\n    ytest,\n    frozen_vr_cohort,\n):\n    rss = PeakRSS().start()\n    wall_start = time.perf_counter()\n\n    governance_time = 0.0\n    ledger_bytes = 0\n    ledger_file = ""\n    governance_df = pd.DataFrame()\n\n    if scenario == "Naïve Centralized":\n        selected = list(client_ids)\n\n    elif scenario == "TADP-AA Centralized":\n        governance_df, policy_accepted, governance_time, ledger_bytes, ledger_file = run_governance(\n            raw_clients, k, seed, scenario\n        )\n        selected = list(client_ids)\n\n    elif scenario == "TADP-VR Centralized":\n        governance_df, selected, governance_time, ledger_bytes, ledger_file = run_governance(\n            raw_clients, k, seed, scenario\n        )\n        if set(selected) != set(frozen_vr_cohort):\n            raise RuntimeError(\n                f"K={k}, seed={seed}: centralized VR cohort differs from frozen VR cohort."\n            )\n\n    else:\n        raise ValueError(scenario)\n\n    if not selected:\n        raise RuntimeError(f"{scenario}: no selected contributors.")\n\n    Xtrain = np.vstack([client_arrays[c][0] for c in selected])\n    ytrain = np.concatenate([client_arrays[c][1] for c in selected])\n\n    per_round_steps = sum(natural_client_steps(client_arrays, c) for c in selected)\n    total_steps = NUM_ROUNDS_FL * per_round_steps\n\n    release_keras_memory()\n    set_seed(pair_initialization_seed(seed, scenario))\n    model = build_model(Xtrain.shape[1])\n\n    training_time = train_exact_steps(\n        model,\n        Xtrain,\n        ytrain,\n        total_steps,\n        pair_initialization_seed(seed, scenario) + 77,\n    )\n\n    metrics = evaluate_model(model, Xtest, ytest)\n    model_params = int(model.count_params())\n\n    wallclock = time.perf_counter() - wall_start\n    rss_info = rss.stop()\n\n    del model, Xtrain, ytrain\n    release_keras_memory()\n\n    return {\n        "selected_clients": list(selected),\n        "selected_count": len(selected),\n        "optimizer_steps": int(total_steps),\n        "training_time_s": float(training_time),\n        "governance_time_s": float(governance_time),\n        "wallclock_s": float(wallclock),\n        "communication_mb": np.nan,\n        "ledger_bytes": int(ledger_bytes),\n        "ledger_file": ledger_file,\n        "model_params": model_params,\n        "metrics": metrics,\n        "rss": rss_info,\n        "governance_df": governance_df,\n        "round_rows": [],\n    }\n\n\n# =============================================================================\n# 13. FEDERATED SCENARIO\n# =============================================================================\n\ndef run_federated(\n    scenario,\n    k,\n    seed,\n    client_ids,\n    raw_clients,\n    client_arrays,\n    Xtest,\n    ytest,\n    frozen_vr_cohort,\n    vr_round_step_target,\n):\n    rss = PeakRSS().start()\n    wall_start = time.perf_counter()\n\n    governance_time = 0.0\n    ledger_bytes = 0\n    ledger_file = ""\n    governance_df = pd.DataFrame()\n\n    if scenario == "Vanilla FedAvg":\n        frozen_selected = list(client_ids)\n\n    elif scenario == "TADP-AA Federated":\n        governance_df, policy_accepted, governance_time, ledger_bytes, ledger_file = run_governance(\n            raw_clients, k, seed, scenario\n        )\n        frozen_selected = list(client_ids)\n\n    elif scenario == "TADP-VR Federated":\n        governance_df, frozen_selected, governance_time, ledger_bytes, ledger_file = run_governance(\n            raw_clients, k, seed, scenario\n        )\n        if set(frozen_selected) != set(frozen_vr_cohort):\n            raise RuntimeError(\n                f"K={k}, seed={seed}: federated VR cohort differs from frozen VR cohort."\n            )\n\n    elif scenario == "Random-K":\n        frozen_selected = None\n\n    else:\n        raise ValueError(scenario)\n\n    input_dim = next(iter(client_arrays.values()))[0].shape[1]\n\n    # One initialization per scenario.\n    release_keras_memory()\n    init_seed = pair_initialization_seed(seed, scenario)\n    set_seed(init_seed)\n\n    local_model, optimizer_initial_state = prepare_reusable_local_model(input_dim)\n    model_params = int(local_model.count_params())\n    model_mb = model_params * 4.0 / (1024 ** 2)\n    global_weights = [w.copy() for w in local_model.get_weights()]\n\n    total_training_time = 0.0\n    total_comm = 0.0\n    total_optimizer_steps = 0\n    round_rows = []\n\n    random_rng = np.random.default_rng(int(seed) + 44001 + int(k))\n\n    for round_idx in range(1, NUM_ROUNDS_FL + 1):\n\n        if scenario == "Random-K":\n            vr_k = len(frozen_vr_cohort)\n            selected = sorted(\n                random_rng.choice(\n                    client_ids,\n                    size=vr_k,\n                    replace=False,\n                ).tolist()\n            )\n\n            natural_map = {\n                cid: natural_client_steps(client_arrays, cid)\n                for cid in selected\n            }\n\n            step_map = allocate_exact_randomk_steps(\n                natural_map,\n                int(vr_round_step_target),\n            )\n\n        else:\n            selected = list(frozen_selected)\n            step_map = {\n                cid: natural_client_steps(client_arrays, cid)\n                for cid in selected\n            }\n\n        # Streaming FedAvg: accumulate n_i * w_i as each client finishes,\n        # rather than retaining every client\'s full model weights in memory.\n        weighted_sum = None\n        total_samples = 0\n        round_train_time = 0.0\n\n        for pos, cid in enumerate(selected):\n            Xc, yc = client_arrays[cid]\n\n            local_seed = (\n                pair_initialization_seed(seed, scenario)\n                + 10000 * round_idx\n                + pos\n            )\n\n            set_seed(local_seed)\n\n            # Same global starting model for this round.\n            local_model.set_weights(global_weights)\n\n            # Fresh Adam state for every client.\n            reset_optimizer_state(local_model, optimizer_initial_state)\n\n            round_train_time += train_exact_steps(\n                local_model,\n                Xc,\n                yc,\n                step_map[cid],\n                local_seed,\n            )\n\n            lw = local_model.get_weights()\n            n_i = int(len(Xc))\n\n            if weighted_sum is None:\n                weighted_sum = [\n                    w.astype(np.float64) * float(n_i)\n                    for w in lw\n                ]\n            else:\n                for j, w in enumerate(lw):\n                    weighted_sum[j] += w.astype(np.float64) * float(n_i)\n\n            total_samples += n_i\n\n            del lw\n            gc.collect()\n\n        if weighted_sum is None or total_samples <= 0:\n            raise RuntimeError(\n                f"{scenario}: no client weights available for aggregation."\n            )\n\n        new_weights = [\n            (acc / float(total_samples)).astype(global_weights[j].dtype)\n            for j, acc in enumerate(weighted_sum)\n        ]\n\n        global_weights = [w.copy() for w in new_weights]\n\n        round_steps = int(sum(step_map.values()))\n        round_comm = float(\n            model_mb\n            * len(selected)\n            * 2.0\n            * (1.0 + PROTOCOL_OVERHEAD)\n        )\n\n        total_training_time += round_train_time\n        total_comm += round_comm\n        total_optimizer_steps += round_steps\n\n        round_rows.append({\n            "experiment_version": EXPERIMENT_VERSION,\n            "K": int(k),\n            "seed": int(seed),\n            "scenario": scenario,\n            "round": int(round_idx),\n            "selected_clients": int(len(selected)),\n            "selected_ids": ",".join(selected),\n            "optimizer_steps": int(round_steps),\n            "communication_mb": float(round_comm),\n        })\n\n        del weighted_sum, new_weights\n        gc.collect()\n\n    # Reuse the same model object for final evaluation.\n    local_model.set_weights(global_weights)\n    metrics = evaluate_model(local_model, Xtest, ytest)\n\n    del local_model, global_weights, optimizer_initial_state\n    release_keras_memory()\n\n    wallclock = time.perf_counter() - wall_start\n    rss_info = rss.stop()\n\n    return {\n        "selected_clients": list(frozen_selected) if frozen_selected is not None else [],\n        "selected_count": len(frozen_selected) if frozen_selected is not None else len(frozen_vr_cohort),\n        "optimizer_steps": int(total_optimizer_steps),\n        "training_time_s": float(total_training_time),\n        "governance_time_s": float(governance_time),\n        "wallclock_s": float(wallclock),\n        "communication_mb": float(total_comm),\n        "ledger_bytes": int(ledger_bytes),\n        "ledger_file": ledger_file,\n        "model_params": model_params,\n        "metrics": metrics,\n        "rss": rss_info,\n        "governance_df": governance_df,\n        "round_rows": round_rows,\n    }\n\n\n# =============================================================================\n# 14. RESULT FILES / RESUME\n# =============================================================================\n\nRAW_RESULTS_CSV = TABLES_DIR / "ExperimentC_raw_results.csv"\nROUND_AUDIT_CSV = TABLES_DIR / "ExperimentC_round_audit.csv"\nGOVERNANCE_AUDIT_CSV = TABLES_DIR / "ExperimentC_governance_audit.csv"\nEVIDENCE_ASSIGNMENT_CSV = TABLES_DIR / "ExperimentC_random_evidence_assignments.csv"\nPARITY_AUDIT_CSV = TABLES_DIR / "ExperimentC_parity_audit.csv"\nSUMMARY_CSV = TABLES_DIR / "ExperimentC_mean_sd.csv"\nMANUSCRIPT_CSV = TABLES_DIR / "ExperimentC_manuscript_scalability_table.csv"\nMETADATA_JSON = TABLES_DIR / "ExperimentC_metadata.json"\n\nif RESUME and RAW_RESULTS_CSV.exists():\n    RESULTS = pd.read_csv(RAW_RESULTS_CSV).to_dict("records")\nelse:\n    RESULTS = []\n\nif RESUME and ROUND_AUDIT_CSV.exists():\n    ROUND_AUDIT = pd.read_csv(ROUND_AUDIT_CSV).to_dict("records")\nelse:\n    ROUND_AUDIT = []\n\nif RESUME and GOVERNANCE_AUDIT_CSV.exists():\n    GOVERNANCE_AUDIT = pd.read_csv(GOVERNANCE_AUDIT_CSV).to_dict("records")\nelse:\n    GOVERNANCE_AUDIT = []\n\n\ndef _compatible_row(row):\n    return str(row.get("experiment_version", "")) == EXPERIMENT_VERSION\n\n\ndef block_complete(k, seed):\n    if not RESULTS:\n        return False\n\n    d = pd.DataFrame(RESULTS)\n    required = {"experiment_version", "K", "seed", "scenario"}\n    if d.empty or not required.issubset(d.columns):\n        return False\n\n    q = d[\n        d["experiment_version"].astype(str).eq(EXPERIMENT_VERSION)\n        & pd.to_numeric(d["K"], errors="coerce").eq(int(k))\n        & pd.to_numeric(d["seed"], errors="coerce").eq(int(seed))\n    ]\n\n    return len(q) == len(SCENARIOS) and set(q["scenario"].astype(str)) == set(SCENARIOS)\n\n\ndef drop_partial_block(k, seed):\n    global RESULTS, ROUND_AUDIT, GOVERNANCE_AUDIT\n\n    def keep(row):\n        try:\n            same = (\n                str(row.get("experiment_version", "")) == EXPERIMENT_VERSION\n                and int(row.get("K")) == int(k)\n                and int(row.get("seed")) == int(seed)\n            )\n            return not same\n        except Exception:\n            return True\n\n    RESULTS = [r for r in RESULTS if keep(r)]\n    ROUND_AUDIT = [r for r in ROUND_AUDIT if keep(r)]\n    GOVERNANCE_AUDIT = [r for r in GOVERNANCE_AUDIT if keep(r)]\n\n\ndef persist():\n    pd.DataFrame(RESULTS).to_csv(RAW_RESULTS_CSV, index=False)\n    pd.DataFrame(ROUND_AUDIT).to_csv(ROUND_AUDIT_CSV, index=False)\n    pd.DataFrame(GOVERNANCE_AUDIT).to_csv(GOVERNANCE_AUDIT_CSV, index=False)\n\n\n# =============================================================================\n# 15. MAIN EXECUTION\n# =============================================================================\n\nexperiment_start = time.perf_counter()\n\nif not POST_ONLY:\n    for k in ACTIVE_KS:\n        print("\\n" + "=" * 96)\n        print(f"K={k}")\n        print("=" * 96)\n\n        for seed in ACTIVE_SEEDS:\n            run_number = RUN_SEEDS.index(seed) + 1\n\n            if RESUME and block_complete(k, seed):\n                print(\n                    f"⏭️ K={k}, seed={seed}: compatible completed block found; skipping."\n                )\n                continue\n\n            drop_partial_block(k, seed)\n\n            print(\n                f"\\n--- K={k} | seed={seed} | "\n                f"run {run_number}/{len(RUN_SEEDS)} ---"\n            )\n            set_seed(seed)\n\n            client_ids, raw_clients, client_arrays, Xtest, ytest = prepare_k_seed(\n                k, seed\n            )\n\n            if WORKER_MODE:\n                try:\n                    del GLOBAL_TRAIN_DF\n                except Exception:\n                    pass\n                try:\n                    del GLOBAL_TEST_DF\n                except Exception:\n                    pass\n                gc.collect()\n\n            # -------------------------------------------------------------\n            # Seeded random evidence assignment — no fixed 50% profile mix.\n            # -------------------------------------------------------------\n            assignment_df = evidence_assignment_table(k, seed)\n            profile_counts = (\n                assignment_df["evidence_profile"]\n                .value_counts()\n                .sort_index()\n                .to_dict()\n            )\n            print("🎲 Random documentary-evidence assignment:")\n            print(f"   mode: {PROFILE_ASSIGNMENT_MODE}")\n            print(\n                f"   assignment seed: "\n                f"{PROFILE_ASSIGNMENT_SEED_BASE + int(seed)}"\n            )\n            print(f"   profile counts: {profile_counts}")\n\n            # Append/replace only this K×seed assignment in the audit CSV.\n            if EVIDENCE_ASSIGNMENT_CSV.exists():\n                try:\n                    old_assign = pd.read_csv(EVIDENCE_ASSIGNMENT_CSV)\n                    if {\n                        "experiment_version", "K", "seed"\n                    }.issubset(old_assign.columns):\n                        keep = ~(\n                            old_assign["experiment_version"].astype(str).eq(\n                                EXPERIMENT_VERSION\n                            )\n                            & pd.to_numeric(\n                                old_assign["K"], errors="coerce"\n                            ).eq(int(k))\n                            & pd.to_numeric(\n                                old_assign["seed"], errors="coerce"\n                            ).eq(int(seed))\n                        )\n                        old_assign = old_assign.loc[keep].copy()\n                    assignment_out = pd.concat(\n                        [old_assign, assignment_df],\n                        ignore_index=True\n                    )\n                except Exception:\n                    assignment_out = assignment_df.copy()\n            else:\n                assignment_out = assignment_df.copy()\n\n            assignment_out.to_csv(EVIDENCE_ASSIGNMENT_CSV, index=False)\n\n            # -------------------------------------------------------------\n            # Freeze the TADP-VR policy once for this K×seed block.\n            # -------------------------------------------------------------\n            (\n                frozen_gov,\n                frozen_vr_cohort,\n                frozen_policy_time,\n                frozen_policy_bytes,\n                frozen_policy_ledger,\n            ) = run_governance(\n                raw_clients,\n                k,\n                seed,\n                "TADP-VR Frozen Policy",\n            )\n\n            if not frozen_vr_cohort:\n                raise RuntimeError(\n                    f"K={k}, seed={seed}: TADP-VR accepted zero clients. "\n                    "The random evidence draw is retained as part of the "\n                    "pre-specified seed rather than silently resampled."\n                )\n\n            if len(frozen_vr_cohort) == k:\n                print(\n                    "⚠️ This randomized block admitted all clients. "\n                    "The result is not altered or resampled; Random-K will "\n                    "therefore coincide in cohort size for this block."\n                )\n\n            vr_acceptance_rate = len(frozen_vr_cohort) / float(k)\n            vr_round_step_target = sum(\n                natural_client_steps(client_arrays, cid)\n                for cid in frozen_vr_cohort\n            )\n\n            print(\n                f"✅ Frozen TADP-VR cohort: "\n                f"{len(frozen_vr_cohort)}/{k} "\n                f"({vr_acceptance_rate:.1%}) "\n                f"| per-round natural steps={vr_round_step_target}"\n            )\n            print(f"   IDs: {frozen_vr_cohort}")\n\n            frozen_audit = frozen_gov.copy()\n            frozen_audit["experiment_version"] = EXPERIMENT_VERSION\n            frozen_audit["audit_source"] = "same_seed_frozen_VR_policy"\n            GOVERNANCE_AUDIT.extend(frozen_audit.to_dict("records"))\n\n            for scenario in SCENARIOS:\n                print(f"\\n▶ {scenario}", flush=True)\n\n                if scenario.endswith("Centralized"):\n                    output = run_centralized(\n                        scenario,\n                        k,\n                        seed,\n                        client_ids,\n                        raw_clients,\n                        client_arrays,\n                        Xtest,\n                        ytest,\n                        frozen_vr_cohort,\n                    )\n                    mode = "centralized"\n\n                else:\n                    output = run_federated(\n                        scenario,\n                        k,\n                        seed,\n                        client_ids,\n                        raw_clients,\n                        client_arrays,\n                        Xtest,\n                        ytest,\n                        frozen_vr_cohort,\n                        vr_round_step_target,\n                    )\n                    mode = "federated"\n\n                m = output["metrics"]\n\n                estimated_training_energy_wh = (\n                    output["training_time_s"] * POWER_W / 3600.0\n                )\n                estimated_e2e_energy_wh = (\n                    output["wallclock_s"] * POWER_W / 3600.0\n                )\n\n                result_row = {\n                    "experiment_version": EXPERIMENT_VERSION,\n                    "K": int(k),\n                    "run": int(run_number),\n                    "seed": int(seed),\n                    "scenario": scenario,\n                    "mode": mode,\n                    "configured_clients": int(k),\n                    "vr_cohort_size": int(len(frozen_vr_cohort)),\n                    "vr_acceptance_rate": float(vr_acceptance_rate),\n                    "selected_clients_summary": int(output["selected_count"]),\n                    "optimizer_steps": int(output["optimizer_steps"]),\n                    "training_time_s": float(output["training_time_s"]),\n                    "governance_time_s": float(output["governance_time_s"]),\n                    "wallclock_s": float(output["wallclock_s"]),\n                    "estimated_training_energy_wh": float(\n                        estimated_training_energy_wh\n                    ),\n                    "estimated_e2e_energy_wh": float(\n                        estimated_e2e_energy_wh\n                    ),\n                    "derived_cost_usd": float(\n                        estimated_e2e_energy_wh\n                        / 1000.0\n                        * COST_USD_PER_KWH\n                    ),\n                    "derived_co2_kg": float(\n                        estimated_e2e_energy_wh\n                        / 1000.0\n                        * CARBON_KG_PER_KWH\n                    ),\n                    "communication_mb": output["communication_mb"],\n                    "ledger_bytes": int(output["ledger_bytes"]),\n                    "ledger_file": output["ledger_file"],\n                    "model_params": int(output["model_params"]),\n                    "rss_start_mb": float(output["rss"]["rss_start_mb"]),\n                    "rss_peak_mb": float(output["rss"]["rss_peak_mb"]),\n                    "rss_peak_delta_mb": float(\n                        output["rss"]["rss_peak_delta_mb"]\n                    ),\n                    "evidence_assignment_mode": PROFILE_ASSIGNMENT_MODE,\n                    "evidence_assignment_seed": int(\n                        PROFILE_ASSIGNMENT_SEED_BASE + int(seed)\n                    ),\n                    **m,\n                }\n\n                RESULTS.append(result_row)\n\n                for rr in output["round_rows"]:\n                    ROUND_AUDIT.append(rr)\n\n                if not output["governance_df"].empty:\n                    g = output["governance_df"].copy()\n                    g["experiment_version"] = EXPERIMENT_VERSION\n                    g["audit_source"] = "scenario_governance"\n                    GOVERNANCE_AUDIT.extend(g.to_dict("records"))\n\n                print(\n                    f"   clients={output[\'selected_count\']} | "\n                    f"steps={output[\'optimizer_steps\']} | "\n                    f"F1={m[\'f1\']:.4f} | "\n                    f"AUC={m[\'roc_auc\']:.4f} | "\n                    f"wall={output[\'wallclock_s\']:.2f}s"\n                )\n\n            persist()\n\n            del raw_clients, client_arrays, Xtest, ytest\n            gc.collect()\n            tf.keras.backend.clear_session()\n\n            print(f"✅ Saved completed K={k}, seed={seed} block.")\n\n# A worker runs exactly one isolated block and stops here.\nif WORKER_MODE:\n    print(\n        f"\\n✅ Fresh-process worker complete: "\n        f"K={ACTIVE_KS[0]}, seed={ACTIVE_SEEDS[0]}"\n    )\n    sys.exit(0)\n\n# =============================================================================\n# 16. STRICT VALIDATION\n# =============================================================================\n\nRESULTS_DF = pd.DataFrame(RESULTS)\nROUNDS_DF = pd.DataFrame(ROUND_AUDIT)\n\n# Keep only this code version for final outputs.\nRESULTS_DF = RESULTS_DF[\n    RESULTS_DF["experiment_version"].astype(str).eq(EXPERIMENT_VERSION)\n].copy()\n\nROUNDS_DF = ROUNDS_DF[\n    ROUNDS_DF["experiment_version"].astype(str).eq(EXPERIMENT_VERSION)\n].copy()\n\nexpected_rows = len(KS) * len(RUN_SEEDS) * len(SCENARIOS)\n\nif len(RESULTS_DF) != expected_rows:\n    raise RuntimeError(\n        f"Experiment C incomplete: expected {expected_rows} scenario rows, "\n        f"found {len(RESULTS_DF)}."\n    )\n\nparity_rows = []\n\nfor k in KS:\n    for seed in RUN_SEEDS:\n\n        block = RESULTS_DF[\n            RESULTS_DF["K"].eq(k)\n            & RESULTS_DF["seed"].eq(seed)\n        ]\n\n        if len(block) != len(SCENARIOS) or set(block["scenario"]) != set(SCENARIOS):\n            raise RuntimeError(f"K={k}, seed={seed}: incomplete scenario set.")\n\n        # ---------------------------------------------------------------------\n        # Full participation: FedAvg and TADP-AA Federated must be K/K.\n        # ---------------------------------------------------------------------\n        for scenario in ["Vanilla FedAvg", "TADP-AA Federated"]:\n            q = ROUNDS_DF[\n                ROUNDS_DF["K"].eq(k)\n                & ROUNDS_DF["seed"].eq(seed)\n                & ROUNDS_DF["scenario"].eq(scenario)\n            ]\n\n            if len(q) != NUM_ROUNDS_FL:\n                raise RuntimeError(f"{scenario}, K={k}, seed={seed}: missing rounds.")\n\n            if not (q["selected_clients"].astype(int) == int(k)).all():\n                raise RuntimeError(\n                    f"{scenario}, K={k}, seed={seed}: full participation failed."\n                )\n\n        # ---------------------------------------------------------------------\n        # VR frozen cohort.\n        # ---------------------------------------------------------------------\n        vr = ROUNDS_DF[\n            ROUNDS_DF["K"].eq(k)\n            & ROUNDS_DF["seed"].eq(seed)\n            & ROUNDS_DF["scenario"].eq("TADP-VR Federated")\n        ].sort_values("round")\n\n        if len(vr) != NUM_ROUNDS_FL:\n            raise RuntimeError(f"TADP-VR, K={k}, seed={seed}: missing rounds.")\n\n        if vr["selected_ids"].nunique() != 1:\n            raise RuntimeError(\n                f"TADP-VR, K={k}, seed={seed}: cohort changed across rounds."\n            )\n\n        # ---------------------------------------------------------------------\n        # Random-K same K and same optimizer-step budget EVERY ROUND.\n        # ---------------------------------------------------------------------\n        rk = ROUNDS_DF[\n            ROUNDS_DF["K"].eq(k)\n            & ROUNDS_DF["seed"].eq(seed)\n            & ROUNDS_DF["scenario"].eq("Random-K")\n        ].sort_values("round")\n\n        if len(rk) != NUM_ROUNDS_FL:\n            raise RuntimeError(f"Random-K, K={k}, seed={seed}: missing rounds.")\n\n        match = vr[\n            ["round", "selected_clients", "optimizer_steps"]\n        ].merge(\n            rk[["round", "selected_clients", "optimizer_steps"]],\n            on="round",\n            suffixes=("_vr", "_rk"),\n        )\n\n        if not (\n            match["selected_clients_vr"].astype(int)\n            == match["selected_clients_rk"].astype(int)\n        ).all():\n            raise RuntimeError(\n                f"Random-K client-count parity failed for K={k}, seed={seed}."\n            )\n\n        if not (\n            match["optimizer_steps_vr"].astype(int)\n            == match["optimizer_steps_rk"].astype(int)\n        ).all():\n            raise RuntimeError(\n                f"Random-K step parity failed for K={k}, seed={seed}."\n            )\n\n        # ---------------------------------------------------------------------\n        # Centralized ↔ paired FL exact optimizer-step parity.\n        # ---------------------------------------------------------------------\n        for cent, fed in PAIR_MAP.items():\n            cent_steps = int(\n                block.loc[block["scenario"].eq(cent), "optimizer_steps"].iloc[0]\n            )\n            fed_steps = int(\n                block.loc[block["scenario"].eq(fed), "optimizer_steps"].iloc[0]\n            )\n\n            parity_rows.append({\n                "experiment_version": EXPERIMENT_VERSION,\n                "K": k,\n                "seed": seed,\n                "centralized": cent,\n                "federated": fed,\n                "centralized_steps": cent_steps,\n                "federated_steps": fed_steps,\n                "parity_ratio": cent_steps / fed_steps if fed_steps else np.nan,\n                "exact_parity": cent_steps == fed_steps,\n            })\n\n            if cent_steps != fed_steps:\n                raise RuntimeError(\n                    f"Step parity failed: K={k}, seed={seed}, "\n                    f"{cent}={cent_steps}, {fed}={fed_steps}"\n                )\n\nPARITY_DF = pd.DataFrame(parity_rows)\nPARITY_DF.to_csv(PARITY_AUDIT_CSV, index=False)\n\nprint("\\n" + "=" * 96)\nprint("✅ ALL EXPERIMENT C VALIDATION GATES PASSED")\nprint("=" * 96)\nprint("✅ Three-class target")\nprint("✅ Full participation baselines")\nprint("✅ TADP-VR frozen cohort")\nprint("✅ Random-K same client count as same-run TADP-VR Federated")\nprint("✅ Random-K exact same optimizer-step budget in every round")\nprint("✅ Centralized↔federated exact optimizer-step parity")\n\n\n# =============================================================================\n# 17. MEAN ± SD SUMMARY\n# =============================================================================\n\nsummary_metrics = [\n    "accuracy",\n    "precision",\n    "recall",\n    "f1",\n    "roc_auc",\n    "training_time_s",\n    "governance_time_s",\n    "wallclock_s",\n    "estimated_training_energy_wh",\n    "estimated_e2e_energy_wh",\n    "communication_mb",\n    "ledger_bytes",\n    "rss_peak_mb",\n    "rss_peak_delta_mb",\n    "optimizer_steps",\n    "vr_cohort_size",\n    "vr_acceptance_rate",\n]\n\nsummary_rows = []\n\nfor (k, scenario), g in RESULTS_DF.groupby(["K", "scenario"], sort=False):\n    row = {\n        "experiment_version": EXPERIMENT_VERSION,\n        "K": int(k),\n        "scenario": scenario,\n        "n_runs": int(g["seed"].nunique()),\n    }\n\n    for metric in summary_metrics:\n        vals = pd.to_numeric(g[metric], errors="coerce").dropna()\n        row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan\n        row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else np.nan\n\n    summary_rows.append(row)\n\nSUMMARY = pd.DataFrame(summary_rows)\nSUMMARY.to_csv(SUMMARY_CSV, index=False)\n\nMANUSCRIPT_COLUMNS = [\n    "K",\n    "scenario",\n    "n_runs",\n    "vr_cohort_size_mean",\n    "vr_acceptance_rate_mean",\n    "vr_acceptance_rate_sd",\n    "accuracy_mean",\n    "accuracy_sd",\n    "f1_mean",\n    "f1_sd",\n    "roc_auc_mean",\n    "roc_auc_sd",\n    "wallclock_s_mean",\n    "wallclock_s_sd",\n    "estimated_e2e_energy_wh_mean",\n    "estimated_e2e_energy_wh_sd",\n    "communication_mb_mean",\n    "communication_mb_sd",\n    "ledger_bytes_mean",\n    "ledger_bytes_sd",\n    "rss_peak_delta_mb_mean",\n    "rss_peak_delta_mb_sd",\n]\n\nSUMMARY[MANUSCRIPT_COLUMNS].to_csv(MANUSCRIPT_CSV, index=False)\n\n\n# =============================================================================\n# 18. FIGURES\n# =============================================================================\n\ndef _plot_metric(mean_col, sd_col, ylabel, title, filename, scenarios=None):\n    scenarios = scenarios or SCENARIOS\n    fig, ax = plt.subplots(figsize=(11, 6))\n\n    for scenario in scenarios:\n        q = SUMMARY[SUMMARY["scenario"].eq(scenario)].sort_values("K")\n        ax.errorbar(\n            q["K"],\n            q[mean_col],\n            yerr=q[sd_col],\n            marker="o",\n            capsize=3,\n            label=scenario,\n        )\n\n    ax.set_xlabel("Number of data contributors (K)")\n    ax.set_ylabel(ylabel)\n    ax.set_title(title)\n    ax.grid(alpha=0.3)\n    ax.legend(frameon=False, fontsize=8, ncol=2)\n    fig.tight_layout()\n    fig.savefig(FIG_DIR / f"{filename}.png", dpi=600, bbox_inches="tight")\n    fig.savefig(FIG_DIR / f"{filename}.pdf", bbox_inches="tight")\n    plt.show()\n    plt.close(fig)\n\n\n_plot_metric(\n    "wallclock_s_mean",\n    "wallclock_s_sd",\n    "End-to-end scenario wall-clock time (s)",\n    "Experiment C — End-to-End Runtime Scalability (mean ± SD, 3 seeds)",\n    "ExperimentC_runtime_vs_K",\n)\n\n_plot_metric(\n    "f1_mean",\n    "f1_sd",\n    "Macro-F1",\n    "Experiment C — Utility Robustness across Client Scale",\n    "ExperimentC_f1_vs_K",\n)\n\n_plot_metric(\n    "communication_mb_mean",\n    "communication_mb_sd",\n    "Modeled FL model-exchange communication (MB)",\n    "Experiment C — Federated Communication Scalability",\n    "ExperimentC_communication_vs_K",\n    scenarios=FED_SCENARIOS,\n)\n\n_plot_metric(\n    "rss_peak_delta_mb_mean",\n    "rss_peak_delta_mb_sd",\n    "Incremental process peak RSS (MB)",\n    "Experiment C — Process-Memory Scalability",\n    "ExperimentC_memory_vs_K",\n)\n\n\n# =============================================================================\n# 19. METADATA\n# =============================================================================\n\nmetadata = {\n    "experiment_version": EXPERIMENT_VERSION,\n    "created_utc": datetime.now(timezone.utc).isoformat(),\n    "study_type": "end-to-end client-scale / federation-size scalability",\n    "fixed_total_dataset": True,\n    "big_data_scalability_claim": False,\n    "K_values": KS,\n    "seeds": RUN_SEEDS,\n    "FL_rounds": NUM_ROUNDS_FL,\n    "local_epochs": LOCAL_EPOCHS,\n    "batch_size": BATCH_SIZE,\n    "dirichlet_alpha": DIRICHLET_ALPHA,\n    "target": "3-class readmitted: NO=0, <30=1, >30=2",\n    "fixed_global_test_set": True,\n    "train_only_scaler": True,\n    "k_dependent_numeric_shift": False,\n    "WAC_min": WAC_MIN,\n    "T_R": GOOD_CUT,\n    "T_A": HIGH_CUT,\n    "random_evidence_assignment_mode": PROFILE_ASSIGNMENT_MODE,\n    "random_evidence_assignment_seed_base": PROFILE_ASSIGNMENT_SEED_BASE,\n    "evidence_profile_sampling": "uniform with replacement from S1-S10; no target admission rate",\n    "nested_profile_assignment_across_K": True,\n    "random_k_source": "same-run TADP-VR Federated cohort",\n    "random_k_exact_client_count_parity": True,\n    "random_k_exact_round_step_parity": True,\n    "centralized_federated_exact_step_parity": True,\n    "centralized_communication": "N/A; raw-data transfer not modeled",\n    "fl_communication_formula": (\n        "FP32 model bytes × participating clients × 2 directions × 1.12"\n    ),\n    "energy_semantics": (\n        "Model-based estimate from measured execution time and fixed POWER_W; "\n        "not hardware-metered."\n    ),\n    "memory_safe_execution": True,\n    "fast_reusable_local_model": True,\n    "streaming_fedavg": True,\n    "memory_note": "One reusable local model per federated scenario; model weights reset to global state and Adam optimizer state reset for every client; streaming weighted FedAvg; tf.data prefetch=1.",\n    "scenarios": SCENARIOS,\n    "total_elapsed_s": float(time.perf_counter() - experiment_start),\n}\n\nwith open(METADATA_JSON, "w", encoding="utf-8") as f:\n    json.dump(metadata, f, indent=2)\n\n\n# =============================================================================\n# 20. FINAL OUTPUT\n# =============================================================================\n\nprint("\\n" + "=" * 96)\nprint("✅ EXPERIMENT C COMPLETE")\nprint("=" * 96)\nprint(f"Raw results:       {RAW_RESULTS_CSV}")\nprint(f"Round audit:       {ROUND_AUDIT_CSV}")\nprint(f"Governance audit:  {GOVERNANCE_AUDIT_CSV}")\nprint(f"Parity audit:      {PARITY_AUDIT_CSV}")\nprint(f"Mean±SD summary:   {SUMMARY_CSV}")\nprint(f"Manuscript table:  {MANUSCRIPT_CSV}")\nprint(f"Figures:           {FIG_DIR}")\nprint(f"Metadata:          {METADATA_JSON}")\nprint(\n    "\\nInterpretation: increasing K with the total dataset held fixed tests "\n    "federation-size/client-scale scalability, not big-data scalability."\n)\n'

def _write_internal_core():
    internal_dir = Path("./experiment_C_end_to_end_v3_5_fast_memsafe_random_evidence_3seeds") / "_internal"
    internal_dir.mkdir(parents=True, exist_ok=True)
    core_file = internal_dir / "_tadp_expc_internal_core_v3_5_fast_memsafe.py"

    # Fail-safe: force the embedded worker to use the exact same seed list as
    # the outer launcher, so launcher/worker seed drift cannot happen again.
    worker_source = CORE_SOURCE

    worker_source = re_sub_seed_list(worker_source, MASTER_SEEDS)
    core_file.write_text(worker_source, encoding="utf-8")
    return core_file


def re_sub_seed_list(source, seeds):
    import re
    replacement = "RUN_SEEDS = " + repr(list(seeds))
    source, n = re.subn(
        r"RUN_SEEDS\s*=\s*\[[^\]]*\]",
        replacement,
        source,
        count=1,
    )
    if n != 1:
        raise RuntimeError(
            "Internal consistency error: could not set worker RUN_SEEDS."
        )
    return source


def _run_block(core_file, k, seed, block_index, total_blocks):
    env = os.environ.copy()
    env["TADP_ONLY_K"] = str(k)
    env["TADP_ONLY_SEED"] = str(seed)
    env.pop("TADP_POST_ONLY", None)

    print("\n" + "=" * 96)
    print(f"BLOCK {block_index}/{total_blocks} — K={k}, seed={seed}")
    print("=" * 96)
    sys.stdout.flush()

    result = subprocess.run(
        [sys.executable, "-u", str(core_file)],
        env=env,
        stdout=None,
        stderr=None,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Experiment C worker failed for K={k}, seed={seed} "
            f"with exit code {result.returncode}. "
            "The worker output immediately above contains the real cause."
        )


def _run_post_validation(core_file):
    env = os.environ.copy()
    env.pop("TADP_ONLY_K", None)
    env.pop("TADP_ONLY_SEED", None)
    env["TADP_POST_ONLY"] = "1"

    print("\n" + "=" * 96)
    print("ALL BLOCKS COMPLETE — FINAL STRICT VALIDATION")
    print("=" * 96)
    sys.stdout.flush()

    result = subprocess.run(
        [sys.executable, "-u", str(core_file)],
        env=env,
        stdout=None,
        stderr=None,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Final Experiment C validation failed. "
            "The worker output immediately above contains the real cause."
        )


def _run_all():
    core_file = _write_internal_core()

    blocks = [
        (k, seed)
        for k in MASTER_KS
        for seed in MASTER_SEEDS
    ]

    # Deterministic random ordering reduces confounding between larger K and
    # long-session runtime/thermal/resource drift.
    rng = np.random.default_rng(20260915)
    rng.shuffle(blocks)

    print("=" * 96)
    print("TADP EXPERIMENT C v3.5 FAST + MEMORY-SAFE — ALL-IN-ONE END-TO-END SCALABILITY")
    print("=" * 96)
    print(f"K values: {MASTER_KS}")
    print(f"Seeds: {MASTER_SEEDS}")
    print("Non-IID partition: Dirichlet alpha = 1.0")
    print("Random evidence assignment: seeded uniform sampling WITH replacement")
    print("Fresh process per K×seed block: YES")
    print("Random-K matched to same-run TADP-VR Federated: YES")
    print("Centralized↔federated exact optimizer-step parity: YES")
    print(f"Total fresh blocks: {len(blocks)}")
    print(f"Total scenario-runs: {len(blocks) * 7}")

    print("\nRandomized execution order:")
    for i, (k, seed) in enumerate(blocks, 1):
        print(f"  {i:2d}. K={k}, seed={seed}")
    print("=" * 96)
    sys.stdout.flush()

    for i, (k, seed) in enumerate(blocks, 1):
        _run_block(core_file, k, seed, i, len(blocks))

    _run_post_validation(core_file)

    print("\n" + "=" * 96)
    print("✅ EXPERIMENT C v3.1 COMPLETE")
    print("=" * 96)


if __name__ == "__main__":
    _run_all()


TADP EXPERIMENT C v3.5 FAST + MEMORY-SAFE — ALL-IN-ONE END-TO-END SCALABILITY
K values: [10, 20, 30, 50]
Seeds: [42, 142, 242]
Non-IID partition: Dirichlet alpha = 1.0
Random evidence assignment: seeded uniform sampling WITH replacement
Fresh process per K×seed block: YES
Random-K matched to same-run TADP-VR Federated: YES
Centralized↔federated exact optimizer-step parity: YES
Total fresh blocks: 12
Total scenario-runs: 84

Randomized execution order:
   1. K=20, seed=242
   2. K=30, seed=142
   3. K=10, seed=42
   4. K=20, seed=142
   5. K=50, seed=242
   6. K=10, seed=142
   7. K=20, seed=42
   8. K=50, seed=142
   9. K=50, seed=42
  10. K=30, seed=42
  11. K=10, seed=242
  12. K=30, seed=242

BLOCK 1/12 — K=20, seed=242

BLOCK 2/12 — K=30, seed=142

BLOCK 3/12 — K=10, seed=42

BLOCK 4/12 — K=20, seed=142

BLOCK 5/12 — K=50, seed=242

BLOCK 6/12 — K=10, seed=142

BLOCK 7/12 — K=20, seed=42

BLOCK 8/12 — K=50, seed=142

BLOCK 9/12 — K=50, seed=42

BLOCK 10/12 — K=30, seed=42

BLOCK 11

In [ ]:
# ============================================================
# DOWNLOAD ALL EXPERIMENT C v3.5 RESULTS
# ============================================================

import os
import shutil
from pathlib import Path

# Look automatically for the Experiment C v3.5 output folder
candidates = list(Path("/content").glob("*Experiment*C*v3*5*"))
candidates += list(Path("/content").glob("experiment_C*v3*5*"))

# Keep directories only
candidates = [p for p in candidates if p.is_dir()]

if not candidates:
    raise FileNotFoundError(
        "Could not find the Experiment C v3.5 results folder in /content. "
        "Make sure Experiment C has finished running."
    )

# If more than one matching folder exists, use the most recently modified one
RESULTS_DIR = max(candidates, key=lambda p: p.stat().st_mtime)

print(f"Found Experiment C results folder:\n{RESULTS_DIR}")

# Create one ZIP file containing the complete results folder
ZIP_BASE = "/content/TADP_ExperimentC_v3_5_ALL_RESULTS"

ZIP_FILE = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=str(RESULTS_DIR)
)

print("\n" + "=" * 80)
print("TADP EXPERIMENT C v3.5 — RESULTS PACKAGE")
print("=" * 80)
print(f"Results folder : {RESULTS_DIR}")
print(f"ZIP file       : {ZIP_FILE}")
print(f"ZIP size       : {os.path.getsize(ZIP_FILE) / (1024**2):.2f} MB")
print("=" * 80)

# Automatically download the ZIP to your laptop
from google.colab import files

print("\nStarting download to your laptop...")
files.download(ZIP_FILE)

Found Experiment C results folder:
/content/experiment_C_end_to_end_v3_5_fast_memsafe_random_evidence_3seeds

TADP EXPERIMENT C v3.5 — RESULTS PACKAGE
Results folder : /content/experiment_C_end_to_end_v3_5_fast_memsafe_random_evidence_3seeds
ZIP file       : /content/TADP_ExperimentC_v3_5_ALL_RESULTS.zip
ZIP size       : 1.85 MB

Starting download to your laptop...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =============================================================================
# EXPERIMENT C v3.5 — REVIEWER POST-ANALYSIS
# NO TRAINING / NO MODEL RERUN
# =============================================================================

import os
import sys
import json
import shutil
import platform
from pathlib import Path

import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# 1. LOCATE EXPERIMENT C v3.5
# -------------------------------------------------------------------------

RESULTS_DIR = Path(
    "/content/experiment_C_end_to_end_v3_5_fast_memsafe_random_evidence_3seeds"
)

if not RESULTS_DIR.exists():
    candidates = [
        p for p in Path("/content").glob("*v3_5*")
        if p.is_dir() and "experiment" in p.name.lower()
    ]

    if not candidates:
        raise FileNotFoundError(
            "Experiment C v3.5 results folder was not found."
        )

    RESULTS_DIR = max(
        candidates,
        key=lambda p: p.stat().st_mtime
    )

TABLES_DIR = RESULTS_DIR / "tables"

RAW_CSV = TABLES_DIR / "ExperimentC_raw_results.csv"
ROUND_CSV = TABLES_DIR / "ExperimentC_round_audit.csv"
PARITY_CSV = TABLES_DIR / "ExperimentC_parity_audit.csv"
META_JSON = TABLES_DIR / "ExperimentC_metadata.json"

required = [
    RAW_CSV,
    ROUND_CSV,
    PARITY_CSV,
    META_JSON,
]

missing = [
    str(p)
    for p in required
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing)
    )

print("=" * 100)
print("EXPERIMENT C v3.5 — CORRECTED FINAL POST-ANALYSIS")
print("=" * 100)
print(f"Results folder:\n{RESULTS_DIR}")

# -------------------------------------------------------------------------
# 2. LOAD RESULTS
# -------------------------------------------------------------------------

raw = pd.read_csv(RAW_CSV)
rounds = pd.read_csv(ROUND_CSV)
parity = pd.read_csv(PARITY_CSV)

with open(META_JSON, "r", encoding="utf-8") as f:
    metadata = json.load(f)

EXPECTED_K = [10, 20, 30, 50]
EXPECTED_SEEDS = [42, 142, 242]

EXPECTED_SCENARIOS = [
    "Naïve Centralized",
    "TADP-AA Centralized",
    "TADP-VR Centralized",
    "Vanilla FedAvg",
    "TADP-AA Federated",
    "TADP-VR Federated",
    "Random-K",
]

FEDERATED_SCENARIOS = [
    "Vanilla FedAvg",
    "TADP-AA Federated",
    "TADP-VR Federated",
    "Random-K",
]

TADP_SCENARIOS = [
    "TADP-AA Centralized",
    "TADP-VR Centralized",
    "TADP-AA Federated",
    "TADP-VR Federated",
]

EXPECTED_ROWS = (
    len(EXPECTED_K)
    * len(EXPECTED_SEEDS)
    * len(EXPECTED_SCENARIOS)
)

# -------------------------------------------------------------------------
# 3. COMPLETENESS
# -------------------------------------------------------------------------

actual_keys = (
    raw[
        ["K", "seed", "scenario"]
    ]
    .drop_duplicates()
)

expected_keys = (
    pd.MultiIndex.from_product(
        [
            EXPECTED_K,
            EXPECTED_SEEDS,
            EXPECTED_SCENARIOS,
        ],
        names=["K", "seed", "scenario"],
    )
    .to_frame(index=False)
)

key_check = expected_keys.merge(
    actual_keys,
    on=["K", "seed", "scenario"],
    how="left",
    indicator=True,
)

missing_runs = key_check[
    key_check["_merge"] != "both"
]

duplicates = (
    raw.groupby(
        ["K", "seed", "scenario"]
    )
    .size()
    .reset_index(name="rows")
)

duplicates = duplicates[
    duplicates["rows"] != 1
]

complete = (
    len(raw) == EXPECTED_ROWS
    and missing_runs.empty
    and duplicates.empty
)

print("\n1. COMPLETENESS")
print("-" * 100)
print(f"Expected scenario-runs : {EXPECTED_ROWS}")
print(f"Observed scenario-runs : {len(raw)}")
print(
    f"Result                 : "
    f"{'PASS' if complete else 'FAIL'}"
)

# -------------------------------------------------------------------------
# 4. CHECK NaN VALUES PROPERLY
# -------------------------------------------------------------------------

print("\n2. MISSING-VALUE CHECK")
print("-" * 100)

core_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "wallclock_s",
    "training_time_s",
    "governance_time_s",
    "ledger_bytes",
    "optimizer_steps",
]

nan_rows = []

for col in core_columns:
    count = int(raw[col].isna().sum())
    nan_rows.append(
        {
            "field": col,
            "unexpected_missing_values": count,
        }
    )

# Communication is different:
# Centralized = N/A by design.
# Federated must contain real communication values.
federated_comm_missing = int(
    raw.loc[
        raw["mode"].str.lower() == "federated",
        "communication_mb"
    ]
    .isna()
    .sum()
)

centralized_comm_values = raw.loc[
    raw["mode"].str.lower() == "centralized",
    "communication_mb"
]

nan_rows.append(
    {
        "field": "communication_mb (federated only)",
        "unexpected_missing_values":
            federated_comm_missing,
    }
)

nan_audit = pd.DataFrame(nan_rows)

display(nan_audit)

unexpected_nan_pass = (
    nan_audit[
        "unexpected_missing_values"
    ].sum()
    == 0
)

print(
    "\nCentralized communication is intentionally "
    "reported as N/A because raw-data transfer "
    "was not modeled."
)

print(
    "Unexpected missing values: "
    + (
        "PASS"
        if unexpected_nan_pass
        else "FAIL"
    )
)

# -------------------------------------------------------------------------
# 5. CENTRALIZED ↔ FEDERATED UPDATE MATCHING
# -------------------------------------------------------------------------

def as_true(x):
    return str(x).strip().lower() in {
        "true", "1", "yes"
    }

parity_pass = (
    len(parity) == 36
    and parity[
        "exact_parity"
    ].map(as_true).all()
)

print("\n3. CENTRALIZED ↔ FEDERATED TRAINING MATCH")
print("-" * 100)

print(f"Expected checks : 36")
print(f"Observed checks : {len(parity)}")

print(
    "Exact local-update matching : "
    + (
        "PASS"
        if parity_pass
        else "FAIL"
    )
)

# -------------------------------------------------------------------------
# 6. TADP-VR ↔ RANDOM-K MATCHED CONTROL
# -------------------------------------------------------------------------

vr_round = rounds[
    rounds["scenario"] == "TADP-VR Federated"
].copy()

rk_round = rounds[
    rounds["scenario"] == "Random-K"
].copy()

matched = vr_round.merge(
    rk_round,
    on=["K", "seed", "round"],
    suffixes=("_VR", "_RandomK"),
    how="outer",
    indicator=True,
)

matched["same_client_count"] = (
    matched["selected_clients_VR"]
    == matched["selected_clients_RandomK"]
)

matched["same_local_updates"] = (
    matched["optimizer_steps_VR"]
    == matched["optimizer_steps_RandomK"]
)

EXPECTED_ROUND_CHECKS = (
    4 * 3 * 4
)

random_match_pass = (
    len(matched) == EXPECTED_ROUND_CHECKS
    and (matched["_merge"] == "both").all()
    and matched["same_client_count"].all()
    and matched["same_local_updates"].all()
)

print("\n4. TADP-VR ↔ RANDOM-K MATCHED CONTROL")
print("-" * 100)

print(
    f"Expected round checks : "
    f"{EXPECTED_ROUND_CHECKS}"
)

print(
    f"Observed round checks : "
    f"{len(matched)}"
)

print(
    "Same number of clients : "
    + (
        "PASS"
        if matched[
            "same_client_count"
        ].all()
        else "FAIL"
    )
)

print(
    "Same local updates     : "
    + (
        "PASS"
        if matched[
            "same_local_updates"
        ].all()
        else "FAIL"
    )
)

# -------------------------------------------------------------------------
# 7. REALIZED ADMISSION RATE
# -------------------------------------------------------------------------

admission_seed = (
    raw.groupby(
        ["K", "seed"],
        as_index=False
    )
    .agg(
        admitted_clients=(
            "vr_cohort_size",
            "first"
        ),
        admission_rate=(
            "vr_acceptance_rate",
            "first"
        ),
    )
)

admission_summary = (
    admission_seed.groupby(
        "K",
        as_index=False
    )
    .agg(
        admitted_clients_mean=(
            "admitted_clients",
            "mean"
        ),
        admitted_clients_sd=(
            "admitted_clients",
            "std"
        ),
        admission_rate_mean=(
            "admission_rate",
            "mean"
        ),
        admission_rate_sd=(
            "admission_rate",
            "std"
        ),
        admission_rate_min=(
            "admission_rate",
            "min"
        ),
        admission_rate_max=(
            "admission_rate",
            "max"
        ),
    )
)

rate_cols = [
    "admission_rate_mean",
    "admission_rate_sd",
    "admission_rate_min",
    "admission_rate_max",
]

admission_summary[
    rate_cols
] *= 100.0

print("\n5. REALIZED TADP-VR ADMISSION")
print("-" * 100)

display(
    admission_summary.round(2)
)

# -------------------------------------------------------------------------
# 8. MAIN PREDICTIVE + RUNTIME SCALABILITY SUMMARY
#
# Communication and ledger are deliberately excluded here.
# They are reported in their own tables below so N/A/zero values
# cannot be misinterpreted.
# -------------------------------------------------------------------------

main_summary = (
    raw.groupby(
        ["K", "scenario"],
        as_index=False
    )
    .agg(
        runs=("seed", "nunique"),

        accuracy_mean=(
            "accuracy",
            "mean"
        ),
        accuracy_sd=(
            "accuracy",
            "std"
        ),

        f1_mean=(
            "f1",
            "mean"
        ),
        f1_sd=(
            "f1",
            "std"
        ),

        roc_auc_mean=(
            "roc_auc",
            "mean"
        ),
        roc_auc_sd=(
            "roc_auc",
            "std"
        ),

        wallclock_s_mean=(
            "wallclock_s",
            "mean"
        ),
        wallclock_s_sd=(
            "wallclock_s",
            "std"
        ),

        training_time_s_mean=(
            "training_time_s",
            "mean"
        ),

        governance_time_s_mean=(
            "governance_time_s",
            "mean"
        ),
    )
)

print("\n6. PREDICTIVE PERFORMANCE AND RUNTIME")
print("-" * 100)

display(
    main_summary.round(4)
)

# -------------------------------------------------------------------------
# 9. FEDERATED COMMUNICATION ONLY
#
# No centralized rows are included because centralized raw-data
# communication was not modeled.
# -------------------------------------------------------------------------

communication_summary = (
    raw[
        raw["scenario"].isin(
            FEDERATED_SCENARIOS
        )
    ]
    .groupby(
        ["K", "scenario"],
        as_index=False
    )
    .agg(
        communication_mb_mean=(
            "communication_mb",
            "mean"
        ),
        communication_mb_sd=(
            "communication_mb",
            "std"
        ),
    )
)

print("\n7. FEDERATED MODEL COMMUNICATION")
print("-" * 100)

print(
    "Centralized communication is not included "
    "because raw-data transfer was not modeled."
)

display(
    communication_summary.round(4)
)

# -------------------------------------------------------------------------
# 10. TADP GOVERNANCE LOG SIZE ONLY
#
# This is NOT total model/system storage.
# It is the size of the pre-training TADP governance JSONL log.
# -------------------------------------------------------------------------

ledger_summary = (
    raw[
        raw["scenario"].isin(
            TADP_SCENARIOS
        )
    ]
    .groupby(
        ["K", "scenario"],
        as_index=False
    )
    .agg(
        governance_log_bytes_mean=(
            "ledger_bytes",
            "mean"
        ),
        governance_log_bytes_sd=(
            "ledger_bytes",
            "std"
        ),
    )
)

ledger_summary[
    "bytes_per_evaluated_client"
] = (
    ledger_summary[
        "governance_log_bytes_mean"
    ]
    / ledger_summary["K"]
)

print("\n8. TADP PRE-TRAINING GOVERNANCE LOG SIZE")
print("-" * 100)

print(
    "This is the TADP governance log only. "
    "It is not model storage or total system storage."
)

display(
    ledger_summary.round(2)
)

# Validate expected baseline zero-ledger behavior.
baseline_scenarios = [
    "Naïve Centralized",
    "Vanilla FedAvg",
    "Random-K",
]

baseline_ledger_zero = (
    raw[
        raw["scenario"].isin(
            baseline_scenarios
        )
    ]["ledger_bytes"]
    .fillna(0)
    .eq(0)
    .all()
)

tadp_ledger_nonzero = (
    raw[
        raw["scenario"].isin(
            TADP_SCENARIOS
        )
    ]["ledger_bytes"]
    .gt(0)
    .all()
)

print(
    "\nBaseline scenarios generate no TADP governance log: "
    + (
        "PASS"
        if baseline_ledger_zero
        else "FAIL"
    )
)

print(
    "TADP scenarios generate a governance log: "
    + (
        "PASS"
        if tadp_ledger_nonzero
        else "FAIL"
    )
)

# -------------------------------------------------------------------------
# 11. TADP-AA vs BASELINE
#
# AA uses exactly the same participating data as the baseline.
# We therefore check whether predictive results are identical rather
# than displaying repeated 0.0000 differences.
#
# We separately report:
# - measured TADP governance-processing time
# - observed overall wall-clock difference
# -------------------------------------------------------------------------

PREDICTIVE_METRICS = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
]

def aa_baseline_check(
    df,
    tadp_scenario,
    baseline_scenario,
    label,
):

    a = df[
        df["scenario"] == tadp_scenario
    ].copy()

    b = df[
        df["scenario"] == baseline_scenario
    ].copy()

    p = a.merge(
        b,
        on=["K", "seed"],
        suffixes=("_AA", "_Baseline"),
    )

    difference_cols = []

    for metric in PREDICTIVE_METRICS:
        col = f"{metric}_abs_difference"

        p[col] = np.abs(
            p[f"{metric}_AA"]
            - p[f"{metric}_Baseline"]
        )

        difference_cols.append(col)

    p["max_predictive_difference"] = (
        p[difference_cols]
        .max(axis=1)
    )

    p[
        "predictive_results_identical"
    ] = (
        p[
            "max_predictive_difference"
        ]
        <= 1e-12
    )

    p["wallclock_difference_s"] = (
        p["wallclock_s_AA"]
        - p["wallclock_s_Baseline"]
    )

    p["wallclock_difference_pct"] = (
        100.0
        * p["wallclock_difference_s"]
        / p["wallclock_s_Baseline"]
    )

    summary = (
        p.groupby(
            "K",
            as_index=False
        )
        .agg(
            measured_governance_time_s_mean=(
                "governance_time_s_AA",
                "mean"
            ),
            measured_governance_time_s_sd=(
                "governance_time_s_AA",
                "std"
            ),

            observed_wallclock_difference_s_mean=(
                "wallclock_difference_s",
                "mean"
            ),
            observed_wallclock_difference_s_sd=(
                "wallclock_difference_s",
                "std"
            ),

            observed_wallclock_difference_pct_mean=(
                "wallclock_difference_pct",
                "mean"
            ),

            maximum_predictive_difference=(
                "max_predictive_difference",
                "max"
            ),

            predictive_results_identical=(
                "predictive_results_identical",
                "all"
            ),
        )
    )

    summary.insert(
        1,
        "comparison",
        label
    )

    return p, summary


aa_c_pairs, aa_c_summary = (
    aa_baseline_check(
        raw,
        "TADP-AA Centralized",
        "Naïve Centralized",
        "AA Centralized vs Naïve Centralized",
    )
)

aa_f_pairs, aa_f_summary = (
    aa_baseline_check(
        raw,
        "TADP-AA Federated",
        "Vanilla FedAvg",
        "AA Federated vs Vanilla FedAvg",
    )
)

aa_summary = pd.concat(
    [
        aa_c_summary,
        aa_f_summary,
    ],
    ignore_index=True,
)

print("\n9. TADP-AA vs GOVERNANCE-OFF BASELINE")
print("-" * 100)

print(
    "The table separates measured governance processing "
    "from normal variation in total wall-clock time."
)

display(
    aa_summary.round(6)
)

# -------------------------------------------------------------------------
# 12. TADP-VR vs FULL FEDAVG
#
# Descriptive trade-off only.
# -------------------------------------------------------------------------

fedavg = raw[
    raw["scenario"] == "Vanilla FedAvg"
].copy()

vr = raw[
    raw["scenario"] == "TADP-VR Federated"
].copy()

vr_fedavg = vr.merge(
    fedavg,
    on=["K", "seed"],
    suffixes=("_VR", "_FedAvg"),
)

vr_fedavg[
    "runtime_reduction_pct"
] = (
    100.0
    * (
        vr_fedavg[
            "wallclock_s_FedAvg"
        ]
        - vr_fedavg[
            "wallclock_s_VR"
        ]
    )
    / vr_fedavg[
        "wallclock_s_FedAvg"
    ]
)

vr_fedavg[
    "communication_reduction_pct"
] = (
    100.0
    * (
        vr_fedavg[
            "communication_mb_FedAvg"
        ]
        - vr_fedavg[
            "communication_mb_VR"
        ]
    )
    / vr_fedavg[
        "communication_mb_FedAvg"
    ]
)

vr_fedavg[
    "f1_difference"
] = (
    vr_fedavg["f1_VR"]
    - vr_fedavg["f1_FedAvg"]
)

tradeoff_summary = (
    vr_fedavg.groupby(
        "K",
        as_index=False
    )
    .agg(
        admission_rate_mean=(
            "vr_acceptance_rate_VR",
            "mean"
        ),

        admission_rate_sd=(
            "vr_acceptance_rate_VR",
            "std"
        ),

        runtime_reduction_pct_mean=(
            "runtime_reduction_pct",
            "mean"
        ),

        runtime_reduction_pct_sd=(
            "runtime_reduction_pct",
            "std"
        ),

        communication_reduction_pct_mean=(
            "communication_reduction_pct",
            "mean"
        ),

        communication_reduction_pct_sd=(
            "communication_reduction_pct",
            "std"
        ),

        f1_difference_mean=(
            "f1_difference",
            "mean"
        ),

        f1_difference_sd=(
            "f1_difference",
            "std"
        ),
    )
)

tradeoff_summary[
    [
        "admission_rate_mean",
        "admission_rate_sd",
    ]
] *= 100.0

print("\n10. TADP-VR vs VANILLA FEDAVG")
print("-" * 100)

print(
    "Positive runtime/communication reduction means "
    "TADP-VR used less time/model communication. "
    "F1 difference is VR minus FedAvg."
)

display(
    tradeoff_summary.round(4)
)

# -------------------------------------------------------------------------
# 13. TADP-VR vs RANDOM-K
#
# Descriptive only here.
# Main statistical comparison = Experiment A with 5 seeds.
# -------------------------------------------------------------------------

random_k = raw[
    raw["scenario"] == "Random-K"
].copy()

vr_random = vr.merge(
    random_k,
    on=["K", "seed"],
    suffixes=("_VR", "_RandomK"),
)

vr_random[
    "f1_difference"
] = (
    vr_random["f1_VR"]
    - vr_random["f1_RandomK"]
)

vr_random[
    "wallclock_difference_s"
] = (
    vr_random["wallclock_s_VR"]
    - vr_random["wallclock_s_RandomK"]
)

vr_random_summary = (
    vr_random.groupby(
        "K",
        as_index=False
    )
    .agg(
        f1_difference_mean=(
            "f1_difference",
            "mean"
        ),

        f1_difference_sd=(
            "f1_difference",
            "std"
        ),

        wallclock_difference_s_mean=(
            "wallclock_difference_s",
            "mean"
        ),

        wallclock_difference_s_sd=(
            "wallclock_difference_s",
            "std"
        ),
    )
)

print("\n11. TADP-VR vs RANDOM-K")
print("-" * 100)

print(
    "Descriptive only. "
    "The 5-seed matched statistical test belongs to Experiment A."
)

display(
    vr_random_summary.round(5)
)

# -------------------------------------------------------------------------
# 14. CLEAR CLIENT-SCALE GROWTH SUMMARY
#
# report quantities that are easier to interpret:
# - value at K=10
# - value at K=50
# - absolute increase
# - percentage increase
# - multiplier
# - average increase for every additional 10 clients
# -------------------------------------------------------------------------

def growth_summary(df, scenario, metric, metric_label):

    d = (
        df[df["scenario"] == scenario]
        .groupby("K", as_index=False)[metric]
        .mean()
        .sort_values("K")
    )

    value_10 = float(
        d.loc[d["K"] == 10, metric].iloc[0]
    )

    value_50 = float(
        d.loc[d["K"] == 50, metric].iloc[0]
    )

    absolute_increase = (
        value_50 - value_10
    )

    percentage_increase = (
        100.0
        * absolute_increase
        / value_10
        if value_10 != 0
        else np.nan
    )

    multiplier = (
        value_50 / value_10
        if value_10 != 0
        else np.nan
    )

    # K increases from 10 to 50 = 40 additional clients,
    # or four groups of 10 clients.
    average_increase_per_10_clients = (
        absolute_increase / 4.0
    )

    return {
        "scenario": scenario,
        "metric": metric_label,
        "K10_mean": value_10,
        "K50_mean": value_50,
        "absolute_increase_10_to_50": absolute_increase,
        "percentage_increase_10_to_50": percentage_increase,
        "multiplier_K50_over_K10": multiplier,
        "average_increase_per_10_clients":
            average_increase_per_10_clients,
    }


growth_rows = []

# Runtime
for scenario in [
    "Vanilla FedAvg",
    "TADP-AA Federated",
    "TADP-VR Federated",
    "Random-K",
]:
    growth_rows.append(
        growth_summary(
            raw,
            scenario,
            "wallclock_s",
            "Runtime (s)"
        )
    )

# Communication
for scenario in [
    "Vanilla FedAvg",
    "TADP-AA Federated",
    "TADP-VR Federated",
    "Random-K",
]:
    growth_rows.append(
        growth_summary(
            raw,
            scenario,
            "communication_mb",
            "Communication (MB)"
        )
    )

# TADP governance log
for scenario in [
    "TADP-AA Federated",
    "TADP-VR Federated",
]:
    growth_rows.append(
        growth_summary(
            raw,
            scenario,
            "ledger_bytes",
            "Governance log (bytes)"
        )
    )

growth_scaling_summary = pd.DataFrame(
    growth_rows
)

print(
    "\n12. CLIENT-SCALE GROWTH FROM K=10 TO K=50"
)
print("-" * 100)

print(
    "This table shows directly how runtime, communication, "
    "and governance-log size changed when the federation "
    "increased from 10 to 50 clients."
)

display(
    growth_scaling_summary.round(3)
)

# -------------------------------------------------------------------------
# 15. MACHINE / SOFTWARE ENVIRONMENT
# -------------------------------------------------------------------------

try:
    import psutil

    total_ram_gb = (
        psutil.virtual_memory().total
        / (1024 ** 3)
    )

except Exception:
    total_ram_gb = None

try:
    import tensorflow as tf
    tensorflow_version = tf.__version__
except Exception:
    tensorflow_version = "Unavailable"

try:
    import sklearn
    sklearn_version = sklearn.__version__
except Exception:
    sklearn_version = "Unavailable"

machine_info = {
    "captured_in_same_live_Experiment_C_session":
        True,

    "platform":
        platform.platform(),

    "operating_system":
        platform.system(),

    "machine":
        platform.machine(),

    "logical_cpu_count":
        os.cpu_count(),

    "total_ram_gb":
        total_ram_gb,

    "python_version":
        sys.version,

    "tensorflow_version":
        tensorflow_version,

    "numpy_version":
        np.__version__,

    "pandas_version":
        pd.__version__,

    "scikit_learn_version":
        sklearn_version,
}

print("\n13. MACHINE / SOFTWARE ENVIRONMENT")
print("-" * 100)

for key, value in machine_info.items():
    print(f"{key}: {value}")

# -------------------------------------------------------------------------
# 16. SAVE CLEAN REVIEWER-READY FILES
# -------------------------------------------------------------------------

POST_DIR = (
    RESULTS_DIR
    / "reviewer_postanalysis_corrected"
)

POST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

nan_audit.to_csv(
    POST_DIR / "01_missing_value_audit.csv",
    index=False,
)

admission_summary.to_csv(
    POST_DIR / "02_admission_summary.csv",
    index=False,
)

main_summary.to_csv(
    POST_DIR / "03_predictive_runtime_summary.csv",
    index=False,
)

communication_summary.to_csv(
    POST_DIR / "04_federated_communication_summary.csv",
    index=False,
)

ledger_summary.to_csv(
    POST_DIR / "05_TADP_governance_log_summary.csv",
    index=False,
)

aa_summary.to_csv(
    POST_DIR / "06_TADP_AA_vs_baseline.csv",
    index=False,
)

tradeoff_summary.to_csv(
    POST_DIR / "07_TADP_VR_vs_FedAvg.csv",
    index=False,
)

vr_random_summary.to_csv(
    POST_DIR / "08_TADP_VR_vs_RandomK.csv",
    index=False,
)

linearity_summary.to_csv(
    POST_DIR / "09_descriptive_linearity.csv",
    index=False,
)

matched[
    [
        "K",
        "seed",
        "round",
        "selected_clients_VR",
        "selected_clients_RandomK",
        "optimizer_steps_VR",
        "optimizer_steps_RandomK",
        "same_client_count",
        "same_local_updates",
    ]
].to_csv(
    POST_DIR / "10_VR_RandomK_matching_audit.csv",
    index=False,
)

with open(
    POST_DIR / "11_machine_environment.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        machine_info,
        f,
        indent=2,
        default=str,
    )

# -------------------------------------------------------------------------
# 17. FINAL VALIDATION
# -------------------------------------------------------------------------

aa_predictive_pass = (
    aa_summary[
        "predictive_results_identical"
    ].all()
)

all_checks_pass = (
    complete
    and unexpected_nan_pass
    and parity_pass
    and random_match_pass
    and baseline_ledger_zero
    and tadp_ledger_nonzero
    and aa_predictive_pass
)

print("\n" + "=" * 100)
print("FINAL CORRECTED EXPERIMENT C VALIDATION")
print("=" * 100)

checks = [
    (
        "84/84 scenario runs",
        complete
    ),
    (
        "No unexpected missing values",
        unexpected_nan_pass
    ),
    (
        "Centralized ↔ federated local updates",
        parity_pass
    ),
    (
        "VR ↔ Random-K client counts",
        matched[
            "same_client_count"
        ].all()
    ),
    (
        "VR ↔ Random-K local updates",
        matched[
            "same_local_updates"
        ].all()
    ),
    (
        "Baseline has no TADP governance log",
        baseline_ledger_zero
    ),
    (
        "TADP scenarios have governance logs",
        tadp_ledger_nonzero
    ),
    (
        "AA predictive results reproduce baseline",
        aa_predictive_pass
    ),
]

for label, passed in checks:
    print(
        f"{label:<48}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("-" * 100)

if all_checks_pass:
    print(
        "✅ Experiment C v3.5 passes all corrected "
        "post-analysis checks."
    )
else:
    print(
        "❌ One or more checks require inspection."
    )

print(
    f"\nCorrected reviewer outputs saved to:\n"
    f"{POST_DIR}"
)

# -------------------------------------------------------------------------
# 18. ZIP + DOWNLOAD
# -------------------------------------------------------------------------

ZIP_BASE = (
    "/content/"
    "TADP_ExperimentC_v3_5_CORRECTED_POSTANALYSIS"
)

ZIP_FILE = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=str(POST_DIR),
)

print(
    f"\nZIP size: "
    f"{os.path.getsize(ZIP_FILE)/(1024**2):.3f} MB"
)

try:
    from google.colab import files

    print(
        "\nStarting download of corrected "
        "post-analysis package..."
    )

    files.download(
        ZIP_FILE
    )

except ImportError:

    print(
        f"\nZIP saved at:\n{ZIP_FILE}"
    )

EXPERIMENT C v3.5 — CORRECTED FINAL POST-ANALYSIS
Results folder:
/content/experiment_C_end_to_end_v3_5_fast_memsafe_random_evidence_3seeds

1. COMPLETENESS
----------------------------------------------------------------------------------------------------
Expected scenario-runs : 84
Observed scenario-runs : 84
Result                 : PASS

2. MISSING-VALUE CHECK
----------------------------------------------------------------------------------------------------


,field,unexpected_missing_values
0,accuracy,0
1,precision,0
2,recall,0
3,f1,0
4,roc_auc,0
5,wallclock_s,0
6,training_time_s,0
7,governance_time_s,0
8,ledger_bytes,0
9,optimizer_steps,0



Centralized communication is intentionally reported as N/A because raw-data transfer was not modeled.
Unexpected missing values: PASS

3. CENTRALIZED ↔ FEDERATED TRAINING MATCH
----------------------------------------------------------------------------------------------------
Expected checks : 36
Observed checks : 36
Exact local-update matching : PASS

4. TADP-VR ↔ RANDOM-K MATCHED CONTROL
----------------------------------------------------------------------------------------------------
Expected round checks : 48
Observed round checks : 48
Same number of clients : PASS
Same local updates     : PASS

5. REALIZED TADP-VR ADMISSION
----------------------------------------------------------------------------------------------------


,K,admitted_clients_mean,admitted_clients_sd,admission_rate_mean,admission_rate_sd,admission_rate_min,admission_rate_max
0,10,5.33,2.31,53.33,23.09,40.00,80.0
1,20,10.67,2.52,53.33,12.58,40.00,65.0
2,30,16.33,2.08,54.44,6.94,46.67,60.0
3,50,26.33,3.06,52.67,6.11,46.00,58.0



6. PREDICTIVE PERFORMANCE AND RUNTIME
----------------------------------------------------------------------------------------------------


...


12. CLIENT-SCALE GROWTH FROM K=10 TO K=50
----------------------------------------------------------------------------------------------------
This table shows directly how runtime, communication, and governance-log size changed when the federation increased from 10 to 50 clients.


,scenario,metric,K10_mean,K50_mean,absolute_increase_10_to_50,percentage_increase_10_to_50,multiplier_K50_over_K10,average_increase_per_10_clients
0,Vanilla FedAvg,Runtime (s),62.303,232.751,170.448,273.579,3.736,42.612
1,TADP-AA Federated,Runtime (s),64.601,235.136,170.535,263.984,3.640,42.634
2,TADP-VR Federated,Runtime (s),39.924,129.834,89.910,225.203,3.252,22.478
3,Random-K,Runtime (s),38.990,127.729,88.739,227.597,3.276,22.185
4,Vanilla FedAvg,Communication (MB),7.723,38.615,30.892,400.000,5.000,7.723
5,TADP-AA Federated,Communication (MB),7.723,38.615,30.892,400.000,5.000,7.723
6,TADP-VR Federated,Communication (MB),4.119,20.337,16.218,393.750,4.938,4.055
7,Random-K,Communication (MB),4.119,20.337,16.218,393.750,4.938,4.055
8,TADP-AA Federated,Governance log (bytes),6395.000,31735.667,25340.667,396.257,4.963,6335.167
9,TADP-VR Federated,Governance log (bytes),6395.000,31735.667,25340.667,396.257,4.963,6335.167



13. MACHINE / SOFTWARE ENVIRONMENT
----------------------------------------------------------------------------------------------------
captured_in_same_live_Experiment_C_session: True
platform: Linux-6.6.122+-x86_64-with-glibc2.39
operating_system: Linux
machine: x86_64
logical_cpu_count: 2
total_ram_gb: 12.671409606933594
python_version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
tensorflow_version: 2.20.0
numpy_version: 2.1.3
pandas_version: 2.2.3
scikit_learn_version: 1.6.1

FINAL CORRECTED EXPERIMENT C VALIDATION
84/84 scenario runs                             : PASS
No unexpected missing values                    : PASS
Centralized ↔ federated local updates           : PASS
VR ↔ Random-K client counts                     : PASS
VR ↔ Random-K local updates                     : PASS
Baseline has no TADP governance log             : PASS
TADP scenarios have governance logs             : PASS
AA predictive results reproduce baseline        : PASS
---------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>